# Hierarchical ReLU-LoRA Spawning — Final Notebook

**Execution order (run top-to-bottom):**
1. Install & model load
2. All class definitions (Hierarchical + DR-LoRA)
3. Jaccard routing diagnostic
4. Shared constants & data loader
5. **Change 1** — Gradient cosine diagnostic *(run before Phase 3)*
6. Phase 2 calibration
7. JSD probe helper
8. Phase 3 config, method factory, eval helper, grid runner
9. Eager spawn ablation
10. **Change 2** — Multi-layer extension *(run after Phase 3)*

> DR-LoRA is available as the `'dr_lora'` method in the Phase 3 grid — uses your teammate's
> full saliency-based implementation (DRLoRALayer / DRLoRATracker / DRLoRAGrowthSchedule).


## Section 1 — Install & Model Load

In [ ]:
# Install (run once per session)
!pip install flash_attn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 66.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done


In [2]:
!pip install transformers>=4.41.0 accelerate datasets peft torch>=2.2.0 tabulate -q

In [1]:
# Cell 9: Load Phi-3.5-mini in bf16 with maximum memory efficiency
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gc

# Clear any lingering VRAM from previous failed runs
torch.cuda.empty_cache()
gc.collect()

MODEL_ID = "microsoft/Phi-tiny-moe-instruct"

print(f"Loading tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model in bf16 with device_map='auto'...")
model_phi = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,   
    device_map='auto',            # FIX: Let accelerate distribute memory safely
    low_cpu_mem_usage=True,       # FIX: Force strict memory management during load
    trust_remote_code=False,      
)

# Enable gradient checkpointing to drastically reduce VRAM during backprop
model_phi.gradient_checkpointing_enable()
model_phi.eval()

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"VRAM reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB")

Loading tokenizer from microsoft/Phi-tiny-moe-instruct...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/325 [00:00<?, ?it/s]

PhimoeForCausalLM LOAD REPORT from: microsoft/Phi-tiny-moe-instruct
Key                                                               | Status     | 
------------------------------------------------------------------+------------+-
model.layers.{0...31}.block_sparse_moe.experts.{0...15}.w1.weight | UNEXPECTED | 
model.layers.{0...31}.block_sparse_moe.experts.{0...15}.w3.weight | UNEXPECTED | 
model.layers.{0...31}.block_sparse_moe.experts.{0...15}.w2.weight | UNEXPECTED | 
model.layers.{0...31}.block_sparse_moe.gate.weight                | UNEXPECTED | 
model.layers.{0...31}.input_layernorm.bias                        | UNEXPECTED | 
model.layers.{0...31}.post_attention_layernorm.bias               | UNEXPECTED | 
model.layers.{0...31}.mlp.experts.gate_up_proj                    | MISSING    | 
model.layers.{0...31}.mlp.router.weight                           | MISSING    | 
model.layers.{0...31}.mlp.experts.down_proj                       | MISSING    | 

Notes:
- UNEXPECTED	:can be i

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Model loaded. VRAM used: 3.8 GB
VRAM reserved: 3.8 GB


## Section 2 — Class Definitions

In [3]:
import torch
import torch.nn as nn

class HierarchicalExpert(nn.Module):
    """
    Low-rank LoRA sub-adapter.  Computes: L(x) = (x A^T B^T) * scaling
    
    Key fix: B is zero-initialized by default. This guarantees that at the
    moment of spawn, the sub-adapter's contribution is exactly zero, which
    is what makes Check 2 (zero-loss-spike) provably pass.
    """
    def __init__(self, in_features, out_features, base_rank=16, lora_alpha=32):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.rank    = base_rank
        self.scaling = lora_alpha / base_rank

        # Standard LoRA init: A ~ N(0, 0.01), B = 0
        # B=0 means output is zero at init → no loss spike on insertion
        self.A = nn.Parameter(torch.randn(base_rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, base_rank))   # FIX: was randn

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Cast A and B to match x's dtype at runtime.
        # This handles bf16/fp16/fp32 transparently regardless of how
        # the parameters were initialized.
        A = self.A.to(x.dtype)
        B = self.B.to(x.dtype)
        return (x @ A.t() @ B.t()) * self.scaling

In [4]:
import torch

class SaturationMonitor:
    """
    Bivariate saturation trigger (§3.2 of proposal).

    Fires when BOTH conditions hold over a rolling window of W steps:
      1. Learning Plateau : |d/dt g_k| < tau_plateau
         where g_k = mean(||B[:,r]|| * ||A[r,:]||) is the rank importance score
      2. High Residual    : EMA(L_expert) > alpha * EMA(L_batch)
    """
    def __init__(self, expert_id: int, window: int = 10,
                 tau_plateau: float = 1e-3, alpha: float = 1.05):
        self.expert_id   = expert_id
        self.window      = window
        self.tau_plateau = tau_plateau
        self.alpha       = alpha
        self.ema_decay   = 0.9

        self._ri_history: list[float] = []
        self._ema_expert: float | None = None
        self._ema_batch:  float | None = None

    # ------------------------------------------------------------------
    @staticmethod
    def _rank_importance(A: torch.Tensor, B: torch.Tensor) -> float:
        """
        Per-rank importance: mean_r( ||B[:,r]|| * ||A[r,:]|| )
        Measures how much each rank contributes to the overall output norm.
        """
        with torch.no_grad():
            col_norms = B.detach().float().norm(dim=0)  # [rank]
            row_norms = A.detach().float().norm(dim=1)  # [rank]
            return (col_norms * row_norms).mean().item()

    # ------------------------------------------------------------------
    def update(self,
               lora_A: torch.Tensor, lora_B: torch.Tensor,
               expert_loss: float,   batch_loss: float,
               grad_A=None,          grad_B=None) -> bool:
        """
        Call once per step. Returns True when spawn should be triggered.
        grad_A / grad_B are unused here but kept for API compatibility.
        """
        d = self.ema_decay
        if self._ema_expert is None:
            self._ema_expert = expert_loss
            self._ema_batch  = batch_loss
        else:
            self._ema_expert = d * self._ema_expert + (1 - d) * expert_loss
            self._ema_batch  = d * self._ema_batch  + (1 - d) * batch_loss

        self._ri_history.append(self._rank_importance(lora_A, lora_B))

        if len(self._ri_history) < self.window:
            return False

        # OLS slope of rank-importance over the window
        recent = self._ri_history[-self.window:]
        x = torch.arange(len(recent), dtype=torch.float32)
        y = torch.tensor(recent,      dtype=torch.float32)
        slope = (
            (x * y).mean() - x.mean() * y.mean()
        ) / (x.var(unbiased=False) + 1e-12)

        plateau  = abs(slope.item()) < self.tau_plateau
        high_err = self._ema_expert > self.alpha * self._ema_batch

        return bool(plateau and high_err)

    def reset_after_spawn(self):
        self._ri_history.clear()
        self._ema_expert = None
        self._ema_batch  = None

In [5]:
# Replace SaturationMonitor entirely for calibration.
# Track two domain-separated loss EMAs instead of a fake proxy.

class ConflictSaturationMonitor:
    """
    Fires when BOTH hold for `window` consecutive steps:
      1. Plateau: rank importance slope < tau_plateau  (LoRA stopped learning)
      2. Conflict: |ema_medical - ema_code| > delta_threshold
                   (two domains have diverged in loss — real entanglement signal)
    """
    def __init__(self, tau_plateau=1e-4, delta_threshold=0.5,
                 window=15, beta=0.9):
        self.tau_plateau      = tau_plateau
        self.delta_threshold  = delta_threshold
        self.window           = window
        self.beta             = beta

        self._ri_history: list = []
        self._ema_code    = None
        self._ema_medical = None
        self._plateau_window  = []
        self._conflict_window = []

    def update(self, lora_A, lora_B, loss_val, domain):
        # ── Update domain-separated EMAs ──────────────────────────────────
        b = self.beta
        if domain == "code":
            self._ema_code = (loss_val if self._ema_code is None
                              else b * self._ema_code + (1-b) * loss_val)
        else:
            self._ema_medical = (loss_val if self._ema_medical is None
                                 else b * self._ema_medical + (1-b) * loss_val)

        # ── Rank importance ───────────────────────────────────────────────
        with torch.no_grad():
            col_norms = lora_B.detach().float().norm(dim=0)
            row_norms = lora_A.detach().float().norm(dim=1)
            ri = (col_norms * row_norms).mean().item()
        self._ri_history.append(ri)

        if len(self._ri_history) < self.window:
            return False

        # OLS slope over window
        recent = self._ri_history[-self.window:]
        x = torch.arange(len(recent), dtype=torch.float32)
        y = torch.tensor(recent, dtype=torch.float32)
        slope = ((x*y).mean() - x.mean()*y.mean()) / (x.var(unbiased=False) + 1e-12)
        plateau = abs(slope.item()) < self.tau_plateau

        # Conflict: both domains seen, and their EMAs have diverged
        if self._ema_code is not None and self._ema_medical is not None:
            conflict = abs(self._ema_medical - self._ema_code) > self.delta_threshold
        else:
            conflict = False

        self._plateau_window.append(plateau)
        self._conflict_window.append(conflict)
        if len(self._plateau_window) > self.window:
            self._plateau_window.pop(0)
            self._conflict_window.pop(0)

        if (len(self._plateau_window) == self.window
                and all(self._plateau_window)
                and all(self._conflict_window)):
            self._plateau_window.clear()
            self._conflict_window.clear()
            return True

        return False

    def reset_after_spawn(self):
        self._ri_history.clear()
        self._plateau_window.clear()
        self._conflict_window.clear()

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HierarchicalPhimoeExperts(nn.Module):
    """
    Drop-in replacement for Phi-MoE's packed expert block.

    The SparseMoeBlock calls: self.experts(hidden_states, top_k_ids, top_k_weights)
    where hidden_states is [T, model_dim].  We intercept this, compute the
    original output, and add per-expert LoRA corrections:

        E_k(x) = W_k x + L_{k,0}(x) + Σ_j ReLU(w_{k,j}^T x) L_{k,j}(x)

    NOTE on dimensions
    ------------------
    down_proj.shape = [num_experts, model_dim, ffn_dim]
                                   ^^^^^^^^^  ^^^^^^^^
                                   out_f      in_f (internal ffn dim)

    hidden_states arriving at the experts is [T, model_dim].
    Our LoRA correction therefore lives in model_dim → model_dim space,
    i.e. lora_in = lora_out = out_f = model_dim.
    Using in_f (ffn_dim) here was the critical shape-mismatch bug.
    """

    def __init__(self, original_experts, base_rank: int = 16, lora_alpha: int = 32):
        super().__init__()
        self.original = original_experts

        # Freeze every original expert parameter
        for p in self.original.parameters():
            p.requires_grad = False

        # Infer packed-weight dimensions
        # down_proj: [num_experts, model_dim, ffn_dim]
        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype

        # LoRA correction lives in model_dim space (input to & output from experts)
        # FIX: was using self.in_f (ffn_dim) — wrong dimension for hidden_states
        self.lora_dim = self.out_f   # = model_dim

        # Base LoRA per expert — registered as nn.ModuleList so optimizer finds them
        dev   = self.original.down_proj.device
        dtype = self.original.down_proj.dtype

        self.base_loras = nn.ModuleList([
            HierarchicalExpert(
                in_features=self.lora_dim,
                out_features=self.lora_dim,
                base_rank=base_rank,
                lora_alpha=lora_alpha,
            ).to(dev).to(dtype)
            for _ in range(self.num_experts)
        ])

        # Spawned adapters: plain Python lists, added to optimizer manually on spawn.
        # spawn_loras[k] = [HierarchicalExpert, ...]
        # spawn_gates[k] = [nn.Parameter of shape [lora_dim], ...]
        self.spawn_loras: list[list] = [[] for _ in range(self.num_experts)]
        self.spawn_gates: list[list] = [[] for _ in range(self.num_experts)]

    # ------------------------------------------------------------------
    @property
    def device(self) -> torch.device:
        return self.original.down_proj.device

    # ------------------------------------------------------------------
    def spawn(self, expert_id: int, rank: int = 8,
              weight_grad: torch.Tensor | None = None) -> list:
        """
        Spawn a new ReLU-gated LoRA sub-adapter inside expert `expert_id`.

        Initialization protocol
        -----------------------
        • A  ← top-r rows of V^H from SVD of grad (residual-gradient init, LoRA-GA style)
                Falls back to small random if grad is None or SVD fails.
        • B  ← zero   (guarantees exact zero output at spawn → no loss spike, Check 2)
        • w  ← N(0, σ²) where σ = 1e-3 · Var(W_k)   (per proposal §3.3)

        Returns list of new nn.Parameters to add to the optimizer.
        """
        dev, dtype = self.device, self.dtype

        lora = HierarchicalExpert(
            in_features=self.lora_dim,
            out_features=self.lora_dim,
            base_rank=rank,
            lora_alpha=2 * rank,
        ).to(dev).to(dtype)

        # Gradient-informed A init, B always stays zero
        if weight_grad is not None:
            try:
                # weight_grad: [lora_dim, lora_dim]
                U, S, Vh = torch.linalg.svd(weight_grad.float(), full_matrices=False)
                with torch.no_grad():
                    lora.A.copy_(Vh[:rank].to(dtype))   # top-r rows of V^H
                    # lora.B stays zero → output = 0 at spawn (Check 2 guaranteed)
            except Exception as e:
                print(f"  [spawn] SVD failed ({e}), using default random A init.")

        # Symmetry-breaking gate: σ = c · Var(W_k)  (proposal eq. 3.3)
        # FIX: original used sqrt(var) inconsistently — using var() matches proposal
        sigma = 1e-3 * self.original.down_proj[expert_id].float().var().item()
        gate = nn.Parameter(
            torch.randn(self.lora_dim, device=dev, dtype=dtype) * sigma
        )

        self.spawn_loras[expert_id].append(lora)
        self.spawn_gates[expert_id].append(gate)

        # Return ALL new leaf parameters so the caller can add them to the optimizer
        return list(lora.parameters()) + [gate]

    # ------------------------------------------------------------------
    def forward(self,
                hidden_states: torch.Tensor,   # [T, model_dim]
                top_k_indices: torch.Tensor,   # [T, top_k]
                top_k_weights: torch.Tensor,   # [T, top_k]
                ) -> torch.Tensor:

        orig_out   = self.original.forward(hidden_states, top_k_indices, top_k_weights)
        # FIX: zeros_like inherits device/dtype automatically — avoids multi-GPU mismatch
        correction = torch.zeros_like(orig_out)

        for k in range(self.num_experts):
            mask = (top_k_indices == k)      # [T, top_k]  bool
            if not mask.any():
                continue

            # Effective routing weight for expert k per token (sum over top_k slots)
            # top_k_indices: [T, top_k], top_k_weights: [T, top_k]
            # Find which top_k slot(s) selected expert k, sum their weights
            eff_w = top_k_weights[:, k]

            # Base LoRA correction
            base_out   = self.base_loras[k](hidden_states)              # [T, lora_dim]
            correction = correction + base_out * eff_w.unsqueeze(-1)

            # Spawned sub-adapter corrections
            for gate_vec, sub_lora in zip(self.spawn_gates[k], self.spawn_loras[k]):
                # ReLU scalar gate per token (proposal §3.1)
                g = F.relu(hidden_states @ gate_vec)                     # [T]
                sub_out    = sub_lora(hidden_states)                     # [T, lora_dim]
                correction = correction + sub_out * (g * eff_w).unsqueeze(-1)

        # Last line of forward(), replacing the current return:
        return (orig_out + correction).to(hidden_states.dtype)

In [7]:
# ── DR-LoRA Layer and Linear wrapper (teammate's implementation) ─────────────
# DRLoRALayer: full-rank capacity reservation + binary rank mask
# DRLoRALinear: wraps original nn.Linear + adds DRLoRALayer on top

import torch
import torch.nn as nn
import math
from typing import Optional

class DRLoRALayer(nn.Module):
    """
    DR-LoRA Layer with Capacity Reservation
    
    Key Innovation: Allocate full rank space (r_max) upfront,
    but activate only a subset (r_init) initially. This avoids
    dynamic tensor resizing and optimizer state corruption.
    """
    
    def __init__(
        self,
        in_features: int,
        out_features: int,
        r_max: int = 16,
        r_init: int = 4,
        lora_alpha: int = 16,
        lora_dropout: float = 0.1,
    ):
        super().__init__()
        
        self.in_features = in_features
        self.out_features = out_features
        self.r_max = r_max
        self.r_init = r_init
        self.lora_alpha = lora_alpha
        # Note: scaling is computed dynamically in forward pass based on active_ranks
        
        # === CAPACITY RESERVATION ===
        # Allocate FULL rank space upfront
        # A ∈ R^(r_max × in_features)
        # B ∈ R^(out_features × r_max)
        self.lora_A = nn.Parameter(torch.zeros(r_max, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r_max))
        
        self.lora_dropout = nn.Dropout(p=lora_dropout)
        
        # === BINARY RANK MASK ===
        # m ∈ {0,1}^r_max
        # 1 = active rank, 0 = dormant rank
        self.register_buffer(
            'rank_mask',
            torch.zeros(r_max, dtype=torch.bool)
        )
        
        # === INITIAL ACTIVATION ===
        # Activate only first r_init ranks
        self.rank_mask[:r_init] = True
        self.active_ranks = r_init
        
        self.reset_parameters()
        
    def reset_parameters(self):
        """Initialize LoRA weights with Kaiming initialization"""
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Masked Forward Pass
        
        Key: Only active ranks contribute to output
        Inactive ranks produce ZERO contribution AND ZERO gradients
        
        Effective weight: W' = W + B[:, active] · A[active, :]
        """
        active_mask = self.rank_mask  # Shape: (r_max,)
        
        if active_mask.any():
            A_active = self.lora_A[active_mask, :]  # (r_active, in_features)
            B_active = self.lora_B[:, active_mask]  # (out_features, r_active)
            
            # Recompute active_ranks from mask to avoid desync after checkpoint load
            active_ranks = active_mask.sum().item()
            # Dynamic scaling based on ACTIVE ranks (not r_max)
            scaling = self.lora_alpha / active_ranks
            
            result = self.lora_dropout(x) @ A_active.T @ B_active.T
            result = result * scaling
        else:
            result = torch.zeros(*x.shape[:-1], self.out_features, device=x.device, dtype=x.dtype)
            
        return result
    
    def activate_rank(self, rank_idx: int):
        """Activate a specific rank (for rank growth)"""
        if rank_idx < self.r_max:
            self.rank_mask[rank_idx] = True
            self.active_ranks = self.rank_mask.sum().item()
    
    def get_active_ranks(self) -> int:
        """Return number of currently active ranks"""
        return self.active_ranks
    
    def get_mask_status(self) -> str:
        """Visual representation of rank mask"""
        mask_str = ''.join(['█' if m else '░' for m in self.rank_mask])
        return f"[{mask_str}] ({self.active_ranks}/{self.r_max})"


class DRLoRALinear(nn.Module):
    """
    Linear layer with DR-LoRA adapter
    Wraps original linear layer + adds DR-LoRA
    """
    
    def __init__(
        self,
        original_layer: nn.Linear,
        r_max: int = 16,
        r_init: int = 4,
        lora_alpha: int = 16,
        lora_dropout: float = 0.1,
    ):
        super().__init__()
        
        self.original_layer = original_layer
        for param in self.original_layer.parameters():
            param.requires_grad = False
        
        self.dr_lora = DRLoRALayer(
            in_features=original_layer.in_features,
            out_features=original_layer.out_features,
            r_max=r_max,
            r_init=r_init,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
        )
        # Move LoRA params to the same device + dtype as the wrapped layer.
        # The base model is already on CUDA/bfloat16 when this wrapper is created;
        # without this the new parameters would stay on CPU in float32.
        _ref = next(original_layer.parameters())
        self.dr_lora.to(device=_ref.device, dtype=_ref.dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_output = self.original_layer(x)
        lora_output = self.dr_lora(x)
        return base_output + lora_output


print('✅ DRLoRALayer + DRLoRALinear defined')


✅ DRLoRALayer + DRLoRALinear defined


In [40]:
# ── DR-LoRA application helpers ──────────────────────────────────────────────
# apply_dr_lora_to_moe_experts: patches model experts with DRLoRALinear wrappers
# freeze_router / unfreeze_routers: router management
# count_parameters: diagnostics

def apply_dr_lora_to_moe_experts(
    model,
    r_max: int = 16,
    r_init: int = 4,
    lora_alpha: int = 16,
    lora_dropout: float = 0.1,
    target_modules: list = None,
):
    """
    Apply DR-LoRA to MoE experts in the model
    
    Args:
        model: The MoE model
        r_max: Maximum rank capacity
        r_init: Initial active ranks
        lora_alpha: LoRA scaling parameter
        lora_dropout: Dropout rate
        target_modules: Which modules to adapt (default: auto-detect or ['gate_proj', 'up_proj', 'down_proj'])
    """
    # Auto-detect target modules if not specified
    if target_modules is None:
        print("Auto-detecting target modules...")
        for layer in model.model.layers:
            if hasattr(layer, 'block_sparse_moe') and hasattr(layer.block_sparse_moe, 'experts'):
                if len(layer.block_sparse_moe.experts) > 0:
                    expert = layer.block_sparse_moe.experts[0]
                    target_modules = [name for name, module in expert.named_children() 
                                     if isinstance(module, nn.Linear)]
                    break
            elif hasattr(layer, 'mlp') and hasattr(layer.mlp, 'experts'):
                # Phi-MoE uses packed experts (PhimoeExperts), not a list
                # It has no individual expert objects to inspect for Linear children
                # Hard-code the known target: the packed down_proj weight
                target_modules = []
                break
        
        if target_modules is None:
            print("⚠ Could not auto-detect modules, using default: ['gate_proj', 'up_proj', 'down_proj']")
            target_modules = ['gate_proj', 'up_proj', 'down_proj']
        else:
            print(f"✓ Auto-detected target modules: {target_modules}")
    
    lora_modules = []
    expert_count = 0
    modules_found = 0
    modules_missed = 0
    
    print(f"\nApplying DR-LoRA to MoE experts...")
    print(f"Target modules: {target_modules}")
    print("-" * 80)
    
    for layer_idx, layer in enumerate(model.model.layers):
        # Check for MoE structure
        if hasattr(layer, 'block_sparse_moe'):
            moe_block = layer.block_sparse_moe
            experts = moe_block.experts
            
            for expert_idx, expert in enumerate(experts):
                expert_count += 1
                
                # Apply DR-LoRA to target modules in each expert
                for module_name in target_modules:
                    if hasattr(expert, module_name):
                        original_layer = getattr(expert, module_name)
                        
                        # Only wrap if it's a Linear layer
                        if isinstance(original_layer, nn.Linear):
                            # Replace with DR-LoRA version
                            dr_lora_layer = DRLoRALinear(
                                original_layer=original_layer,
                                r_max=r_max,
                                r_init=r_init,
                                lora_alpha=lora_alpha,
                                lora_dropout=lora_dropout,
                            )
                            
                            setattr(expert, module_name, dr_lora_layer)
                            lora_modules.append({
                                'layer': layer_idx,
                                'expert': expert_idx,
                                'module': module_name,
                                'dr_lora': dr_lora_layer.dr_lora,
                            })
                            modules_found += 1
                        else:
                            modules_missed += 1
                    else:
                        modules_missed += 1
        
        elif hasattr(layer, 'mlp') and hasattr(layer.mlp, 'experts'):
            moe_block = layer.mlp
            experts = moe_block.experts
            
            for expert_idx, expert in enumerate(experts):
                expert_count += 1
                
                for module_name in target_modules:
                    if hasattr(expert, module_name):
                        original_layer = getattr(expert, module_name)
                        
                        # Only wrap if it's a Linear layer
                        if isinstance(original_layer, nn.Linear):
                            dr_lora_layer = DRLoRALinear(
                                original_layer=original_layer,
                                r_max=r_max,
                                r_init=r_init,
                                lora_alpha=lora_alpha,
                                lora_dropout=lora_dropout,
                            )
                            
                            setattr(expert, module_name, dr_lora_layer)
                            lora_modules.append({
                                'layer': layer_idx,
                                'expert': expert_idx,
                                'module': module_name,
                                'dr_lora': dr_lora_layer.dr_lora,
                            })
                            modules_found += 1
                        else:
                            modules_missed += 1
                    else:
                        modules_missed += 1
    
    print(f"✓ Applied DR-LoRA to {len(lora_modules)} modules across {expert_count} experts")
    if modules_missed > 0:
        print(f"⚠ Warning: {modules_missed} target modules not found or not Linear layers")
    if len(lora_modules) == 0:
        print(f"\n✗ CRITICAL: No modules were wrapped!")
        print(f"   Please check that target_modules matches actual expert module names")
        print(f"   Run the expert structure inspection cell to see available modules")
    return lora_modules


def freeze_router(model):
    """
    Freeze router/gate parameters
    
    WHY? Early router decisions are unstable. If router adapts immediately:
    - routing shifts randomly
    - importance estimates become meaningless
    
    Strategy: First stabilize LoRA directions, then fine-tune router
    """
    frozen_params = 0
    routers_frozen = 0
    
    print("Freezing router parameters...")
    print("-" * 80)
    
    for layer_idx, layer in enumerate(model.model.layers):
        # Check for router/gate in MoE blocks
        if hasattr(layer, 'block_sparse_moe'):
            moe_block = layer.block_sparse_moe
            
            if hasattr(moe_block, 'gate'):
                for param in moe_block.gate.parameters():
                    param.requires_grad = False
                    frozen_params += param.numel()
                routers_frozen += 1
                
            elif hasattr(moe_block, 'router'):
                for param in moe_block.router.parameters():
                    param.requires_grad = False
                    frozen_params += param.numel()
                routers_frozen += 1
        
        elif hasattr(layer, 'mlp') and hasattr(layer.mlp, 'gate'):
            for param in layer.mlp.gate.parameters():
                param.requires_grad = False
                frozen_params += param.numel()
            routers_frozen += 1
            
        elif hasattr(layer, 'mlp') and hasattr(layer.mlp, 'router'):
            for param in layer.mlp.router.parameters():
                param.requires_grad = False
                frozen_params += param.numel()
            routers_frozen += 1
    
    print(f"✓ Frozen {routers_frozen} routers ({frozen_params:,} parameters)")
    return routers_frozen


def count_parameters(model):
    """Count trainable vs frozen parameters"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params
    
    return {
        'total': total_params,
        'trainable': trainable_params,
        'frozen': frozen_params,
        'trainable_pct': 100 * trainable_params / total_params if total_params > 0 else 0,
    }


print('✅ DR-LoRA application helpers defined')


✅ DR-LoRA application helpers defined


In [9]:
# ── DRLoRAGrowthSchedule ─────────────────────────────────────────────────────

import pandas as pd
from tabulate import tabulate

class DRLoRAGrowthSchedule:
    """
    DR-LoRA Growth Scheduling System
    
    Converts rank growth into a scheduled resource allocation problem:
    - When can ranks grow? (growth events)
    - How many ranks per event? (quota)
    - Which experts get them? (fairness over time)
    """
    
    def __init__(
        self,
        total_steps: int,
        warmup_steps: int,
        growth_interval: int,
        r_init: int,
        r_target: int,
        r_max: int,
        num_experts_per_layer: int,
        num_layers: int,
        end_buffer_steps: int = 1000,
    ):
        """
        Initialize growth schedule
        
        Args:
            total_steps: Total training steps (t_end)
            warmup_steps: Warmup period before first growth (t_warmup)
            growth_interval: Steps between growth events (T_grow)
            r_init: Initial active ranks
            r_target: Target final ranks per expert
            r_max: Maximum rank capacity
            num_experts_per_layer: Number of experts per layer
            num_layers: Number of layers with experts
            end_buffer_steps: Stop growth this many steps before end
        """
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.growth_interval = growth_interval
        self.r_init = r_init
        self.r_target = r_target
        self.r_max = r_max
        self.num_experts_per_layer = num_experts_per_layer
        self.num_layers = num_layers
        self.end_buffer_steps = end_buffer_steps
        
        # === VALIDATION ===
        self._validate_config()
        
        # === COMPUTE GROWTH SCHEDULE ===
        
        # 3.1 Growth Events: T_events = floor((t_end - t_warmup) / T_grow)
        # But also respect end buffer
        effective_end = total_steps - end_buffer_steps
        growth_duration = effective_end - warmup_steps
        
        if growth_duration <= 0:
            raise ValueError(
                f"No room for growth! warmup={warmup_steps} + buffer={end_buffer_steps} "
                f">= total_steps={total_steps}"
            )
        
        self.num_growth_events = max(0, growth_duration // growth_interval)
        
        # Compute actual growth event steps
        self.growth_steps = [
            warmup_steps + (i + 1) * growth_interval 
            for i in range(self.num_growth_events)
        ]
        # O(1) lookup set for can_grow_at_step / get_event_index
        self.growth_steps_set = set(self.growth_steps)
        
        # 3.2 Rank Budget: Q = N × (r_target - r_init) / T_events
        self.total_ranks_to_grow = num_experts_per_layer * (r_target - r_init)
        
        if self.num_growth_events > 0:
            self.ranks_per_event = self.total_ranks_to_grow / self.num_growth_events
        else:
            self.ranks_per_event = 0
        
        # Track actual quota per event (may vary slightly due to rounding)
        self.event_quotas = self._compute_event_quotas()
        
        # 3.3 Growth Window
        self.growth_start = warmup_steps
        self.growth_end = effective_end
        
    def _validate_config(self):
        """Validate configuration parameters"""
        if self.r_init >= self.r_target:
            raise ValueError(f"r_init ({self.r_init}) must be < r_target ({self.r_target})")
        
        if self.r_target > self.r_max:
            raise ValueError(f"r_target ({self.r_target}) cannot exceed r_max ({self.r_max})")
        
        if self.warmup_steps >= self.total_steps:
            raise ValueError(f"warmup_steps ({self.warmup_steps}) must be < total_steps ({self.total_steps})")
        
        if self.growth_interval <= 0:
            raise ValueError(f"growth_interval must be positive, got {self.growth_interval}")
        
        if self.num_experts_per_layer <= 0:
            raise ValueError(f"num_experts_per_layer must be positive")
    
    def _compute_event_quotas(self):
        """
        Compute rank quota for each growth event
        
        Distributes total_ranks_to_grow across events,
        handling rounding to ensure exact total.
        """
        if self.num_growth_events == 0:
            return []
        
        quotas = []
        remaining_ranks = self.total_ranks_to_grow
        
        for event_idx in range(self.num_growth_events):
            remaining_events = self.num_growth_events - event_idx
            
            # Distribute evenly, but handle rounding
            quota = remaining_ranks // remaining_events
            quotas.append(quota)
            remaining_ranks -= quota
        
        return quotas
    
    def can_grow_at_step(self, current_step: int) -> bool:
        """
        Check if current step is a valid growth event
        
        Args:
            current_step: Current training step
            
        Returns:
            True if growth is allowed at this step
        """
        # Must be in growth window
        if current_step < self.growth_start or current_step >= self.growth_end:
            return False
        
        # Must be exactly at a growth step
        return current_step in self.growth_steps_set
    
    def get_event_index(self, current_step: int) -> int:
        """
        Get the event index for current step
        
        Args:
            current_step: Current training step
            
        Returns:
            Event index (0-based), or -1 if not a growth step
        """
        if current_step in self.growth_steps_set:
            return self.growth_steps.index(current_step)
        return -1
    
    def get_rank_quota(self, current_step: int = None, event_idx: int = None) -> int:
        """
        Get rank quota for a growth event
        
        Args:
            current_step: Current training step (optional if event_idx provided)
            event_idx: Event index (optional if current_step provided)
            
        Returns:
            Number of ranks to distribute at this event
        """
        if event_idx is None:
            if current_step is None:
                raise ValueError("Must provide either current_step or event_idx")
            event_idx = self.get_event_index(current_step)
        
        if event_idx < 0 or event_idx >= len(self.event_quotas):
            return 0
        
        return self.event_quotas[event_idx]
    
    def get_next_growth_step(self, current_step: int) -> int:
        """
        Get the next growth step after current_step
        
        Args:
            current_step: Current training step
            
        Returns:
            Next growth step, or -1 if no more growth events
        """
        for step in self.growth_steps:
            if step > current_step:
                return step
        return -1
    
    def get_progress(self, current_step: int) -> dict:
        """
        Get progress information for current step
        
        Args:
            current_step: Current training step
            
        Returns:
            Dict with progress metrics
        """
        # Count completed events
        completed_events = sum(1 for step in self.growth_steps if step <= current_step)
        
        # Sum ranks distributed so far
        ranks_distributed = sum(self.event_quotas[:completed_events])
        
        return {
            'current_step': current_step,
            'completed_events': completed_events,
            'total_events': self.num_growth_events,
            'ranks_distributed': ranks_distributed,
            'total_ranks_to_grow': self.total_ranks_to_grow,
            'progress_pct': 100 * ranks_distributed / self.total_ranks_to_grow if self.total_ranks_to_grow > 0 else 100,
            'in_growth_window': self.growth_start <= current_step < self.growth_end,
            'next_growth_step': self.get_next_growth_step(current_step),
        }
    
    def get_schedule_summary(self) -> pd.DataFrame:
        """
        Get summary of entire growth schedule
        
        Returns:
            DataFrame with schedule information
        """
        if self.num_growth_events == 0:
            return pd.DataFrame({
                'Event': [],
                'Step': [],
                'Rank Quota': [],
                'Cumulative Ranks': [],
                'Progress %': [],
            })
        
        data = []
        cumulative = 0
        
        for event_idx, (step, quota) in enumerate(zip(self.growth_steps, self.event_quotas)):
            cumulative += quota
            data.append({
                'Event': event_idx + 1,
                'Step': step,
                'Rank Quota': quota,
                'Cumulative Ranks': cumulative,
                'Progress %': f"{100 * cumulative / self.total_ranks_to_grow:.1f}%",
            })
        
        return pd.DataFrame(data)
    
    def __repr__(self):
        return (
            f"DRLoRAGrowthSchedule(\n"
            f"  total_steps={self.total_steps}, warmup={self.warmup_steps}\n"
            f"  growth_interval={self.growth_interval}, end_buffer={self.end_buffer_steps}\n"
            f"  r_init={self.r_init} → r_target={self.r_target} (capacity={self.r_max})\n"
            f"  experts={self.num_experts_per_layer} × layers={self.num_layers}\n"
            f"  num_events={self.num_growth_events}, ranks_per_event={self.ranks_per_event:.1f}\n"
            f"  growth_window=[{self.growth_start}, {self.growth_end})\n"
            f")"
        )


print('✅ DRLoRAGrowthSchedule defined')


✅ DRLoRAGrowthSchedule defined


In [10]:
# ── DRLoRATracker + unfreeze_routers ─────────────────────────────────────────

import torch.nn.functional as F
from typing import Dict, List, Tuple


class DRLoRATracker:
    """
    Track routing frequency (Eq. 5) and rank importance (Eq. 6-8) with EMA.
    Uses forward hooks to capture actual router softmax weights automatically.
    """

    def __init__(
        self,
        num_layers: int,
        num_experts_per_layer: int,
        ema_beta: float = 0.9,
        device: str = "cuda",
    ):
        self.num_layers = num_layers
        self.num_experts_per_layer = num_experts_per_layer
        self.ema_beta = ema_beta
        self.device = device

        # f^(l,i): routing frequency EMA
        self.routing_frequency = torch.zeros(
            num_layers, num_experts_per_layer, device=device, dtype=torch.float32
        )
        # g^(l,i): rank importance EMA
        self.rank_importance = torch.zeros(
            num_layers, num_experts_per_layer, device=device, dtype=torch.float32
        )

        # Forward hook state
        self._routing_hooks = []
        self._current_router_weights: Dict[int, torch.Tensor] = {}

        self.num_updates = 0

    # ── Routing hooks ──────────────────────────────────────────────────────────

    def register_routing_hooks(self, model) -> int:
        """
        Register forward hooks on gate/router modules to capture softmax weights.
        Hooks fire during each forward pass and populate _current_router_weights.
        Call once after model setup.
        """
        self._remove_routing_hooks()

        for layer_idx, layer in enumerate(model.model.layers):
            gate = None
            if hasattr(layer, "block_sparse_moe"):
                moe = layer.block_sparse_moe
                gate = getattr(moe, "gate", None) or getattr(moe, "router", None)
            elif hasattr(layer, "mlp"):
                gate = getattr(layer.mlp, "gate", None) or getattr(layer.mlp, "router", None)

            if gate is not None:
                def make_hook(idx):
                    def hook(module, inp, output):
                        # output may be logits (batch*seq, N) or tuple
                        logits = output[0] if isinstance(output, tuple) else output
                        if logits.dim() == 3:
                            logits = logits.view(-1, logits.size(-1))
                        # Softmax over experts → average over tokens
                        weights = torch.softmax(logits.float(), dim=-1).mean(dim=0).detach()
                        self._current_router_weights[idx] = weights
                    return hook

                handle = gate.register_forward_hook(make_hook(layer_idx))
                self._routing_hooks.append(handle)

        print(f"Registered routing hooks on {len(self._routing_hooks)} gate modules")
        return len(self._routing_hooks)

    def _remove_routing_hooks(self):
        for h in self._routing_hooks:
            h.remove()
        self._routing_hooks = []
        self._current_router_weights = {}

    # ── EMA updates ────────────────────────────────────────────────────────────

    def update_routing_frequency_from_hooks(self):
        """
        EMA update of routing frequency from data captured by forward hooks (Eq. 5).
        Call after each forward pass (before zero_grad).
        """
        with torch.no_grad():
            for layer_idx, weights in self._current_router_weights.items():
                if layer_idx >= self.num_layers:
                    continue
                if weights.shape[0] != self.num_experts_per_layer:
                    continue
                # f_t = beta * f_{t-1} + (1-beta) * w_t
                self.routing_frequency[layer_idx] = (
                    self.ema_beta * self.routing_frequency[layer_idx]
                    + (1 - self.ema_beta) * weights.to(self.device)
                )
        self._current_router_weights = {}

    def compute_rank_importance(
        self,
        lora_modules: List[Dict],
    ) -> Dict[Tuple[int, int], float]:
        """
        Compute per-expert rank importance from gradients.

        Implements Eq. 6 (per-rank sensitivity score):
            s_j = ||grad_a_j ⊙ a_j||_1  ×  ||grad_b_j ⊙ b_j||_1

        And Eq. 8 (expert-level aggregation):
            g_{l,i} = (1 / r) * sum_j  s_j

        Returns:
            Dict mapping (layer_idx, expert_idx) -> importance score
        """
        importance_scores: Dict[Tuple[int, int], float] = {}

        with torch.no_grad():
            for lora_info in lora_modules:
                layer_idx = lora_info["layer"]
                expert_idx = lora_info["expert"]
                dr_lora = lora_info["dr_lora"]

                if dr_lora.lora_A.grad is None or dr_lora.lora_B.grad is None:
                    continue

                active_mask = dr_lora.rank_mask
                if not active_mask.any():
                    continue

                # A[active]: shape (r_active, in_features)
                A_grad  = dr_lora.lora_A.grad[active_mask]
                A_param = dr_lora.lora_A.data[active_mask]

                # B[active]: shape (out_features, r_active)
                B_grad  = dr_lora.lora_B.grad[:, active_mask]
                B_param = dr_lora.lora_B.data[:, active_mask]

                # Eq. 6: L1 norm of element-wise product per rank dimension
                A_sensitivity = (A_grad * A_param).abs().sum(dim=1)   # (r_active,)
                B_sensitivity = (B_grad * B_param).abs().sum(dim=0)   # (r_active,)

                # Multiplicative aggregation (joint A-B contribution)
                per_rank_scores = A_sensitivity * B_sensitivity        # (r_active,)

                # Eq. 8: average over active ranks
                expert_importance = per_rank_scores.mean().item()

                # Accumulate across modules within same expert (gate/up/down proj)
                key = (layer_idx, expert_idx)
                importance_scores[key] = importance_scores.get(key, 0.0) + expert_importance

        return importance_scores

    def update_rank_importance(self, importance_scores: Dict[Tuple[int, int], float]):
        """EMA update of rank importance (Eq. 7): g_t = beta*g_{t-1} + (1-beta)*s_t"""
        with torch.no_grad():
            for (layer_idx, expert_idx), score in importance_scores.items():
                if layer_idx >= self.num_layers or expert_idx >= self.num_experts_per_layer:
                    continue
                self.rank_importance[layer_idx, expert_idx] = (
                    self.ema_beta * self.rank_importance[layer_idx, expert_idx]
                    + (1 - self.ema_beta) * score
                )
        self.num_updates += 1

    def reset_rank_importance(self, layer_idx: int = None):
        """
        Reset rank importance after a growth event (Algorithm 1, line 13).
        Preserves routing_frequency so well-used experts remain competitive.
        """
        if layer_idx is None:
            self.rank_importance.zero_()
        else:
            self.rank_importance[layer_idx].zero_()

    # ── Scoring ────────────────────────────────────────────────────────────────

    def get_saliency_scores(
        self,
        layer_idx: int,
        lora_modules: List[Dict] = None,
        gamma: float = 0.5,
    ) -> torch.Tensor:
        """
        Saliency score per expert in a layer (Eq. 9):
            S_{l,i} = f_{l,i} * g_{l,i} / (r_{l,i} + 1)^gamma

        Args:
            layer_idx: Which layer to score
            lora_modules: LoRA module list for rank counts (optional)
            gamma: Rank penalty exponent

        Returns:
            Saliency scores tensor (num_experts,)
        """
        f = self.routing_frequency[layer_idx]
        g = self.rank_importance[layer_idx]

        ranks = torch.ones(self.num_experts_per_layer, device=self.device)
        if lora_modules is not None:
            for info in lora_modules:
                if info["layer"] == layer_idx:
                    e = info["expert"]
                    if e < self.num_experts_per_layer:
                        ranks[e] = info["dr_lora"].get_active_ranks()

        return f * g / (ranks + 1).pow(gamma)

    def get_expert_scores(self, layer_idx: int = None) -> torch.Tensor:
        """
        Combined f*g score (without rank penalty) — used for diagnostics.

        Returns:
            (num_layers, num_experts) or (num_experts,) if layer specified
        """
        scores = self.routing_frequency * self.rank_importance
        if layer_idx is not None:
            return scores[layer_idx]
        return scores

    def get_top_experts(self, k: int, layer_idx: int = None) -> List[Tuple[int, int]]:
        """Top-k experts by f*g score."""
        scores = self.get_expert_scores(layer_idx)

        if layer_idx is not None:
            topk = torch.topk(scores, min(k, scores.numel())).indices
            return [(layer_idx, i.item()) for i in topk]
        else:
            flat = scores.flatten()
            topk = torch.topk(flat, min(k, flat.numel())).indices
            N = self.num_experts_per_layer
            return [(i.item() // N, i.item() % N) for i in topk]

    def get_stats(self) -> dict:
        return {
            "num_updates": self.num_updates,
            "avg_routing_freq": self.routing_frequency.mean().item(),
            "max_routing_freq": self.routing_frequency.max().item(),
            "min_routing_freq": self.routing_frequency.min().item(),
            "avg_rank_importance": self.rank_importance.mean().item(),
            "max_rank_importance": self.rank_importance.max().item(),
            "min_rank_importance": self.rank_importance.min().item(),
        }


def unfreeze_routers(model):
    """Unfreeze router parameters after warmup (Algorithm 1, line 5-7)."""
    unfrozen_params = 0
    routers_unfrozen = 0

    print("Unfreezing router parameters...")
    print("-" * 80)

    for layer in model.model.layers:
        moe = (
            getattr(layer, "block_sparse_moe", None)
            or getattr(layer, "mlp", None)
        )
        if moe is None:
            continue
        gate = getattr(moe, "gate", None) or getattr(moe, "router", None)
        if gate is not None:
            for param in gate.parameters():
                param.requires_grad = True
                unfrozen_params += param.numel()
            routers_unfrozen += 1

    print(f"Unfrozen {routers_unfrozen} routers ({unfrozen_params:,} parameters)")
    return routers_unfrozen


print('✅ DRLoRATracker + unfreeze_routers defined')


✅ DRLoRATracker + unfreeze_routers defined


In [11]:
# ── perform_rank_growth + get_rank_distribution ──────────────────────────────

import math
from collections import defaultdict


def perform_rank_growth(
    tracker: DRLoRATracker,
    schedule: DRLoRAGrowthSchedule,
    lora_modules: list,
    current_step: int,
    r_init: int,
    r_max: int,
    p_grow: float = 0.5,
    gamma: float = 0.5,
    verbose: bool = False,
) -> dict:
    """
    Periodic rank growth — Algorithm 1, lines 9-14.

    For each layer independently:
      1. Compute saliency S = f * g / (r+1)^gamma  (Eq. 9)
      2. Sort experts by saliency (descending)
      3. Greedy allocation with quota Q             (Eq. 12)
      4. Activate n_grow dormant rank slots
      5. Reset g_{l,i}=0, preserve f_{l,i}

    Args:
        tracker:       DRLoRATracker with current EMA state
        schedule:      DRLoRAGrowthSchedule for quota lookup
        lora_modules:  List of DR-LoRA module info dicts
        current_step:  Current training step
        r_init:        Initial rank (used to compute r_free, fixed per Eq. 12)
        r_max:         Maximum rank capacity
        p_grow:        Max growth fraction per expert per event
        gamma:         Rank penalty exponent
        verbose:       Print per-expert growth details

    Returns:
        Dict with growth statistics, or {'grew': False} if not a growth step.
    """
    if not schedule.can_grow_at_step(current_step):
        return {"grew": False}

    event_idx   = schedule.get_event_index(current_step)
    Q_per_layer = schedule.get_rank_quota(event_idx=event_idx)

    # r_free is FIXED at init time (Eq. 12 uses initial free capacity, not current)
    r_free         = r_max - r_init
    max_per_expert = max(1, math.floor(r_free * p_grow))   # floor(r_free * p_grow)

    total_new_ranks    = 0
    layers_grown       = 0
    num_experts_grown  = 0
    expert_growth      = {}   # (layer_idx, expert_idx) -> n_new_ranks

    # Group modules by layer, then by expert
    by_layer_expert = defaultdict(lambda: defaultdict(list))
    for info in lora_modules:
        by_layer_expert[info["layer"]][info["expert"]].append(info)

    for layer_idx in sorted(by_layer_expert.keys()):
        layer_experts = by_layer_expert[layer_idx]

        # ── 1. Saliency (Eq. 9) ───────────────────────────────────────────────
        saliency = {}
        for expert_idx, mods in layer_experts.items():
            f     = tracker.routing_frequency[layer_idx, expert_idx].item()
            g     = tracker.rank_importance[layer_idx, expert_idx].item()
            r_cur = mods[0]["dr_lora"].get_active_ranks()
            saliency[expert_idx] = f * g / ((r_cur + 1) ** gamma)

        # ── 2. Sort by saliency (descending) ──────────────────────────────────
        sorted_experts = sorted(saliency, key=lambda e: saliency[e], reverse=True)

        # ── 3. Greedy allocation ───────────────────────────────────────────────
        Q_remain   = Q_per_layer
        layer_grew = False

        for expert_idx in sorted_experts:
            if Q_remain <= 0:
                break

            mods    = layer_experts[expert_idx]
            dl_ref  = mods[0]["dr_lora"]                        # representative module
            r_avail = dl_ref.r_max - dl_ref.get_active_ranks()  # currently inactive ranks

            if r_avail == 0:
                continue

            # Eq. 12: n_grow = min(floor(r_free*p_grow), r_avail_i, Q_remain)
            n_grow = min(max_per_expert, r_avail, Q_remain)
            if n_grow <= 0:
                continue

            # Activate n_grow new rank slots in EVERY module of this expert
            # (gate_proj, up_proj, down_proj all grow together)
            cur_active = dl_ref.get_active_ranks()
            for info in mods:
                dl = info["dr_lora"]
                for j in range(cur_active, cur_active + n_grow):
                    dl.activate_rank(j)

            Q_remain         -= n_grow
            total_new_ranks  += n_grow
            expert_growth[(layer_idx, expert_idx)] = n_grow
            layer_grew       = True
            num_experts_grown += 1

            if verbose:
                print(f"    L{layer_idx:02d} E{expert_idx:02d}: "
                      f"+{n_grow} ranks ({cur_active}->{cur_active+n_grow})  "
                      f"S={saliency[expert_idx]:.6f}")

        # ── 4. Reset rank importance, preserve routing frequency ───────────────
        # (Algorithm 1 line 13)
        tracker.reset_rank_importance(layer_idx=layer_idx)

        if layer_grew:
            layers_grown += 1

    return {
        "grew":             True,
        "step":             current_step,
        "event_idx":        event_idx,
        "Q_per_layer":      Q_per_layer,
        "total_new_ranks":  total_new_ranks,
        "layers_grown":     layers_grown,
        "num_experts_grown": num_experts_grown,
        "expert_growth":    expert_growth,
    }


def get_rank_distribution(lora_modules: list) -> dict:
    """Current rank distribution statistics across all DR-LoRA modules."""
    if not lora_modules:
        return {}
    ranks = [info["dr_lora"].get_active_ranks() for info in lora_modules]
    active_params = sum(
        info["dr_lora"].get_active_ranks()
        * (info["dr_lora"].in_features + info["dr_lora"].out_features)
        for info in lora_modules
    )
    return {
        "min":           min(ranks),
        "max":           max(ranks),
        "mean":          sum(ranks) / len(ranks),
        "active_params": active_params,
    }


print('✅ perform_rank_growth + get_rank_distribution defined')


✅ perform_rank_growth + get_rank_distribution defined


In [12]:
# ── Data preparation and DR-LoRA evaluation helpers ─────────────────────────

from datasets import load_dataset

# ══════════════════════════════════════════════════════════════════════════════
# MASTER SWITCH
# True  → quick validation : 10 train / 3 test  (code path check, < 1 min)
# False → full experiment  : 4 000 train / 500 test  (~3–4 h on T4)
TEST_MODE = False
# ══════════════════════════════════════════════════════════════════════════════

# Full-mode dataset sizes (only used when TEST_MODE = False)
# Fits within a single 4-hour Kaggle GPU session on a T4:
#   4 000 samples × 3 epochs / batch 4  =  3 000 steps  ≈  3–4 h
N_TRAIN_FULL = 4_000
N_TEST_FULL  =   500   # 12.5 % of training; enough for stable accuracy estimate

import re


def extract_answer(text: str) -> str:
    """
    Extract the final numerical answer from a MetaMathQA response.
    Tries multiple patterns in order of specificity.
    """
    # 1. LaTeX \boxed{...}
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].strip().replace(",", "")

    # 2. "The answer is X" (case-insensitive)
    m = re.search(r"the answer is[:\s]+([+-]?\d[\d,]*(?:\.\d+)?)", text, re.I)
    if m:
        return m.group(1).replace(",", "")

    # 3. "= X" at end of last sentence
    m = re.search(r"=\s*([+-]?\d[\d,]*(?:\.\d+)?)\s*[.\n]?\s*$", text)
    if m:
        return m.group(1).replace(",", "")

    # 4. Last number in text as fallback
    nums = re.findall(r"[+-]?\d[\d,]*(?:\.\d+)?", text)
    if nums:
        return nums[-1].replace(",", "")

    return text.strip()


def evaluate_dr_lora(model, tokenizer, test_samples, max_new_tokens=512, max_length=512):
    """
    Generate answers on test samples and compute exact-match accuracy.
    """
    model.eval()
    results = []

    with torch.no_grad():
        for sample in test_samples:
            prompt_msgs = [{"role": "user", "content": sample["query"]}]
            try:
                prompt_text = tokenizer.apply_chat_template(
                    prompt_msgs, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                prompt_text = f"Question: {sample['query']}\nAnswer: "

            inputs = tokenizer(
                prompt_text, return_tensors="pt",
                truncation=True, max_length=max_length,
            ).to(model.device)

            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            new_ids   = out_ids[0, inputs["input_ids"].shape[1]:]
            generated = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

            pred_ans  = extract_answer(generated)
            true_ans  = extract_answer(sample["response"])
            correct   = (pred_ans == true_ans)

            results.append({
                "query":     sample["query"],
                "true_resp": sample["response"],
                "generated": generated,
                "true_ans":  true_ans,
                "pred_ans":  pred_ans,
                "correct":   correct,
            })

    model.train()
    return results


print('✅ prepare_batch + extract_answer + evaluate_dr_lora defined')


✅ prepare_batch + extract_answer + evaluate_dr_lora defined


## Section 3 — Architecture Inspection (optional)

In [13]:
# Cell 10: Inspect Phi-3.5-MoE architecture to find the right module path
# Run this to understand the exact structure before patching.

print("=" * 60)
print("Model architecture inspection")
print("=" * 60)

# Print top-level structure
print("\nTop-level modules:")
for name, module in model_phi.named_children():
    print(f"  {name}: {type(module).__name__}")

# Find the MoE layers
print("\n\nSearching for MoE layer structure...")
moe_layers_found = []
for name, module in model_phi.named_modules():
    class_name = type(module).__name__
    if 'Expert' in class_name or 'MoE' in class_name or 'expert' in name.lower():
        depth = name.count('.')
        if depth < 5:  # Avoid printing nested noise
            print(f"  [{depth}] {name}: {class_name}")
            moe_layers_found.append((name, module))

# Print one expert's structure in detail
print("\n\nDetailed structure of first expert found:")
if moe_layers_found:
    name, module = moe_layers_found[0]
    print(f"  Path: {name}")
    for pname, param in module.named_parameters():
        print(f"    {pname}: {param.shape} dtype={param.dtype}")

Model architecture inspection

Top-level modules:
  model: PhimoeModel
  lm_head: Linear


Searching for MoE layer structure...
  [4] model.layers.0.mlp.experts: PhimoeExperts
  [4] model.layers.1.mlp.experts: PhimoeExperts
  [4] model.layers.2.mlp.experts: PhimoeExperts
  [4] model.layers.3.mlp.experts: PhimoeExperts
  [4] model.layers.4.mlp.experts: PhimoeExperts
  [4] model.layers.5.mlp.experts: PhimoeExperts
  [4] model.layers.6.mlp.experts: PhimoeExperts
  [4] model.layers.7.mlp.experts: PhimoeExperts
  [4] model.layers.8.mlp.experts: PhimoeExperts
  [4] model.layers.9.mlp.experts: PhimoeExperts
  [4] model.layers.10.mlp.experts: PhimoeExperts
  [4] model.layers.11.mlp.experts: PhimoeExperts
  [4] model.layers.12.mlp.experts: PhimoeExperts
  [4] model.layers.13.mlp.experts: PhimoeExperts
  [4] model.layers.14.mlp.experts: PhimoeExperts
  [4] model.layers.15.mlp.experts: PhimoeExperts
  [4] model.layers.16.mlp.experts: PhimoeExperts
  [4] model.layers.17.mlp.experts: PhimoeExperts


## Section 4 — Jaccard Routing Diagnostic

Run before any training to select TARGET_LAYER.

In [14]:
# ── Diagnostics Cell: Jaccard Routing Overlap ────────────────────────────────
# Run BEFORE training to select the best domain conflict pair.
# Uses already-loaded model_phi — no second model load needed.

import torch
from collections import defaultdict
from datasets import load_dataset

# ── Config: edit these to try different pairs ────────────────────────────────
TOP_K          = 2     # Phi-MoE uses top-2 routing (NOT 8 like OLMoE)
N_EXAMPLES     = 60
MAX_LENGTH     = 128   # Keep short to avoid OOM during diagnostic
# Jaccard threshold recalibrated for top-2 routing across 16 experts.
# With top-2, random overlap baseline is ~(2/16)=0.125, so 0.35 is meaningful.
JACCARD_THRESHOLD = 0.35

DATASET_OPTION_A = {
    "name": "Python vs Medical",
    "primary":          ("openai/openai_humaneval", "prompt",        None),
    "conflict":         ("qiaojin/PubMedQA",        "question",      "pqa_labeled"),
}
DATASET_OPTION_B = {
    "name": "Math vs Creative Fiction",
    "primary":          ("DigitalLearningGmbH/MATH-lighteval",           "problem",       "default"),
    "conflict":         ("roneneldan/TinyStories",   "text",          None),
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def load_text_examples(dataset_name, text_col, config_name=None, n=60, preferred_split="train"):
    """Load examples, automatically falling back to whatever split exists."""
    kwargs = {}
    if config_name:
        kwargs["name"] = config_name

    # Load without specifying split first so we can inspect what's available
    ds_dict = load_dataset(dataset_name, **kwargs)
    
    # Pick split: prefer 'train', then 'test', then whatever exists
    available = list(ds_dict.keys())
    if preferred_split in available:
        split = preferred_split
    elif "test" in available:
        split = "test"
    else:
        split = available[0]
    
    print(f"    (using split='{split}', available={available})")
    ds = ds_dict[split]
    return [str(ex[text_col]) for ex in ds.select(range(min(n, len(ds))))]


def get_top_k_experts_for_domain(model, tokenizer, examples, top_k=2,
                                  n_examples=50, max_length=128):
    """
    Returns normalized frequency dict: (layer_idx, expert_idx) → float.
    Normalized so frequencies sum to 1.0 per domain — comparable across domains.
    """
    expert_counts = defaultdict(int)
    total_activations = 0
    model.eval()

    with torch.no_grad():
        for text in examples[:n_examples]:
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
                padding=False,
            ).to(model.device)

            outputs = model(**inputs, output_router_logits=True, return_dict=True)

            if outputs.router_logits is None:
                raise ValueError("router_logits is None")

            for layer_idx, logits in enumerate(outputs.router_logits):
                if logits is None:
                    continue
                if logits.dim() == 3:
                    logits = logits.view(-1, logits.shape[-1])

                top_experts = logits.topk(top_k, dim=-1).indices
                for expert_idx in top_experts.flatten().tolist():
                    expert_counts[(layer_idx, expert_idx)] += 1
                    total_activations += 1

    # Normalize to frequency distribution
    return {k: v / total_activations for k, v in expert_counts.items()}


def compute_weighted_overlap(freq_a: dict, freq_b: dict,
                              concentration_threshold: float = 0.002) -> tuple[float, dict]:
    """
    Only consider experts that are meaningfully concentrated in at least one domain.
    concentration_threshold: minimum frequency in either domain to count the expert.
    
    Returns (jaccard_of_concentrated_experts, per_layer_stats).
    
    With 16 experts and top-2, uniform frequency per layer ≈ 2/16 = 0.125 per layer.
    Across N layers, uniform per (layer, expert) ≈ 0.125/N.
    threshold=0.002 filters out uniformly-distributed experts.
    """
    all_keys = set(freq_a.keys()) | set(freq_b.keys())
    
    # Only keep experts that are non-trivially active in at least one domain
    concentrated = {
        k for k in all_keys
        if freq_a.get(k, 0) >= concentration_threshold
        or freq_b.get(k, 0) >= concentration_threshold
    }
    
    # Of those, find which ones both domains share
    shared = {
        k for k in concentrated
        if freq_a.get(k, 0) >= concentration_threshold
        and freq_b.get(k, 0) >= concentration_threshold
    }
    
    jaccard = len(shared) / len(concentrated) if concentrated else 0.0
    
    # Per-layer breakdown — useful for picking TARGET_LAYER
    layer_stats = defaultdict(lambda: {"shared": 0, "total": 0})
    for (layer_idx, expert_idx) in concentrated:
        layer_stats[layer_idx]["total"] += 1
        if (layer_idx, expert_idx) in shared:
            layer_stats[layer_idx]["shared"] += 1
    
    return jaccard, dict(layer_stats)


# ── Main diagnostic ───────────────────────────────────────────────────────────
def run_jaccard_diagnostic(model, tokenizer, concentration_threshold=0.002):
    print("=" * 65)
    print("JACCARD ROUTING OVERLAP DIAGNOSTIC  (Phi-MoE, top-2)")
    print(f"Threshold: {JACCARD_THRESHOLD}  |  concentration_cutoff: {concentration_threshold}")
    print("=" * 65)

    results     = {}
    layer_stats = {}

    for opt in [DATASET_OPTION_A, DATASET_OPTION_B]:
        name = opt["name"]
        print(f"\n── {name} ──")

        primary_ds,  primary_col,  primary_cfg  = opt["primary"]
        conflict_ds, conflict_col, conflict_cfg  = opt["conflict"]

        print(f"  Loading primary  : {primary_ds}")
        primary_texts  = load_text_examples(primary_ds,  primary_col,  primary_cfg)
        print(f"  Loading conflict : {conflict_ds}")
        conflict_texts = load_text_examples(conflict_ds, conflict_col, conflict_cfg)

        print(f"  Running routing probe...")
        freq_primary  = get_top_k_experts_for_domain(
            model, tokenizer, primary_texts,  top_k=TOP_K, n_examples=N_EXAMPLES)
        freq_conflict = get_top_k_experts_for_domain(
            model, tokenizer, conflict_texts, top_k=TOP_K, n_examples=N_EXAMPLES)

        jaccard, lstats = compute_weighted_overlap(
            freq_primary, freq_conflict, concentration_threshold)
        results[name]     = jaccard
        layer_stats[name] = lstats

        print(f"  Weighted Jaccard : {jaccard:.3f}  "
              f"({'✅ above' if jaccard >= JACCARD_THRESHOLD else '❌ below'} "
              f"threshold {JACCARD_THRESHOLD})")
        
        # Print per-layer overlap — helps choose TARGET_LAYER
        print(f"  Per-layer overlap (shared/total concentrated experts):")
        for layer_idx in sorted(lstats.keys()):
            s = lstats[layer_idx]["shared"]
            t = lstats[layer_idx]["total"]
            bar = "█" * s + "░" * (t - s)
            print(f"    Layer {layer_idx:2d}: {bar}  {s}/{t}")

    # Decision
    print("\n" + "=" * 65)
    print("DECISION")
    print("=" * 65)
    best_name = max(results, key=results.get)
    best_val  = results[best_name]
    for name, j in results.items():
        marker = "◀ SELECTED" if name == best_name else ""
        print(f"  {name:35s}  Jaccard={j:.3f}  {marker}")

    print()
    if best_val >= JACCARD_THRESHOLD:
        print(f"✅ Use: {best_name}")
        # Recommend the layer with highest overlap as TARGET_LAYER
        best_layer = max(
            layer_stats[best_name],
            key=lambda l: layer_stats[best_name][l]["shared"]
        )
        print(f"   Recommended TARGET_LAYER: {best_layer}  "
              f"(most shared expert activations)")
    else:
        best_layer = 15  # fallback
        print(f"⚠️  No pair clears threshold. Using best available: {best_name}")
        print(f"   Falling back to TARGET_LAYER={best_layer}")

    print("=" * 65)
    return results, layer_stats, best_layer


results, layer_stats, recommended_layer = run_jaccard_diagnostic(model_phi, tokenizer)
print(f"\nSet TARGET_LAYER = {recommended_layer} in Cell 11")

JACCARD ROUTING OVERLAP DIAGNOSTIC  (Phi-MoE, top-2)
Threshold: 0.35  |  concentration_cutoff: 0.002

── Python vs Medical ──
  Loading primary  : openai/openai_humaneval


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

    (using split='test', available=['test'])
  Loading conflict : qiaojin/PubMedQA


...

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

    (using split='train', available=['train', 'validation'])
  Running routing probe...
  Weighted Jaccard : 0.341  (❌ below threshold 0.35)
  Per-layer overlap (shared/total concentrated experts):
    Layer  0: ████░░░░  4/8
    Layer  1: ████░░░░░░  4/10
    Layer  2: ████░░░  4/7
    Layer  3: ████░  4/5
    Layer  4: ███░░░░░░  3/9
    Layer  5: ████  4/4
    Layer  6: ██░░░░░░░  2/9
    Layer  7: ███░░░░░  3/8
    Layer  8: ███░░░░░  3/8
    Layer  9: █░░░░░░░  1/8
    Layer 10: ██░░░░░  2/7
    Layer 11: ███░░░░  3/7
    Layer 12: ██░░░░░░░░  2/10
    Layer 13: █░░░░░░░░░  1/10
    Layer 14: ████░░░░  4/8
    Layer 15: ██░░░  2/5
    Layer 16: ███░░░░░  3/8
    Layer 17: █░░░░░░░░  1/9
    Layer 18: ██░░░░░  2/7
    Layer 19: ████░░░░  4/8
    Layer 20: █░░░░░░░░░  1/10
    Layer 21: ███░░░  3/6
    Layer 22: ████░░  4/6
    Layer 23: ██░░░░░░░  2/9
    Layer 24: ██░░░░░  2/7
    Layer 25: ███░░░░  3/7
    Layer 26: ██░░░░░░  2/8
    Layer 27: ██░░░░░░  2/8
    Layer 28: ████░░░░

## Section 5 — Shared Constants & Data Loader

In [15]:
# ── Shared calibration constants (run this before any calibration cell) ───────
BASE_RANK    = 16
LR           = 2e-5
TARGET_LAYER = 0
TARGET_EXPERT = 0
CALIB_STEPS  = 300

In [16]:
# ── Real Data Loader (adapted from dataset_builder.py, notebook-safe) ────────
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset

# ── Dataset config (inline — no config file needed in notebook) ───────────────
DATASET_CFG = {
    "name":         "Python vs Medical",
    "primary":      ("openai/openai_humaneval", "prompt",   None),
    "conflict":     ("qiaojin/PubMedQA",        "question", "pqa_labeled"),
    "domain_names": ["code", "medical"],
}

# Calibration-specific settings — shorter than full training
MAX_LENGTH_CALIB  = 128    # keep short to fit memory during 9-run grid
N_EACH            = 300    # examples per domain for calibration
CONFLICT_RATIO    = 0.5    # severe conflict — most likely to trigger spawning

def load_split_auto(dataset_name, config_name=None, preferred="train"):
    """Load dataset, auto-selecting split (handles HumanEval test-only etc.)."""
    kwargs = {"name": config_name} if config_name else {}
    ds_dict = load_dataset(dataset_name, **kwargs)
    available = list(ds_dict.keys())
    split = preferred if preferred in available else ("test" if "test" in available else available[0])
    print(f"    [{dataset_name}] using split='{split}' (available: {available})")
    return ds_dict[split]


def build_calibration_dataloader(tokenizer, cfg=DATASET_CFG,
                                  conflict_ratio=CONFLICT_RATIO,
                                  max_length=MAX_LENGTH_CALIB,
                                  n_each=N_EACH):
    """
    Interleave primary + conflict domain examples at given ratio.
    Returns a plain list of tokenized dicts (no DataLoader overhead needed
    for single-sample calibration stepping).
    """
    primary_name, primary_col, primary_cfg = cfg["primary"]
    conflict_name, conflict_col, conflict_cfg = cfg["conflict"]

    print("Loading primary dataset...")
    primary_ds = load_split_auto(primary_name, primary_cfg)
    primary_ds = primary_ds.select(range(min(n_each, len(primary_ds))))

    print("Loading conflict dataset...")
    conflict_ds = load_split_auto(conflict_name, conflict_cfg)
    conflict_ds = conflict_ds.select(range(min(n_each, len(conflict_ds))))

    # Extract text
    primary_texts  = [str(ex[primary_col])  for ex in primary_ds]
    conflict_texts = [str(ex[conflict_col]) for ex in conflict_ds]

    # Interleave at conflict_ratio: for every N conflict examples,
    # include (1-ratio)/ratio primary examples
    n_conflict = int(n_each * conflict_ratio)
    n_primary  = n_each - n_conflict
    primary_texts  = primary_texts[:n_primary]
    conflict_texts = conflict_texts[:n_conflict]

    # Build interleaved list: alternate to create domain shift pattern
    import math
    texts, domains = [], []
    ratio = n_conflict / max(n_primary, 1)
    ci = pi = 0
    conflict_acc = 0.0
    while pi < len(primary_texts) or ci < len(conflict_texts):
        # Add primary
        if pi < len(primary_texts):
            texts.append(primary_texts[pi]); domains.append("code"); pi += 1
        # Add conflict proportionally
        conflict_acc += ratio
        while conflict_acc >= 1.0 and ci < len(conflict_texts):
            texts.append(conflict_texts[ci]); domains.append("medical"); ci += 1
            conflict_acc -= 1.0

    print(f"\nDataset built: {len(texts)} total "
          f"({domains.count('code')} code / {domains.count('medical')} medical)")

    # Tokenize
    encodings = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )

    return encodings, domains   # return both so we know domain at each step


# Build once, reuse across all 9 calibration runs
print("Building calibration dataset...")
calib_encodings, calib_domains = build_calibration_dataloader(tokenizer)
print("Done.")

Building calibration dataset...
Loading primary dataset...
    [openai/openai_humaneval] using split='test' (available: ['test'])
Loading conflict dataset...
    [qiaojin/PubMedQA] using split='train' (available: ['train'])

Dataset built: 300 total (150 code / 150 medical)
Done.


## Section 6 — Change 1: Gradient Cosine Similarity Diagnostic

**Run this before Phase 3**, on the clean unpatched model.
Directly measures whether code/medical gradients are anti-aligned in the shared LoRA.
Result determines whether the 'destructive interference' framing is empirically supported.


In [18]:
# ── Inline probe texts (defined here so this cell runs before Phase 3) ────────
# These are reused by the JSD probe cell later — no duplication needed there.
EVAL_CODE_TEXTS = [
    "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
    "def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[0]\n    return quicksort([x for x in arr[1:] if x < pivot]) + [pivot] + quicksort([x for x in arr[1:] if x >= pivot])",
    "import numpy as np\nx = np.array([1, 2, 3])\nprint(np.dot(x, x))",
    "class Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, x):\n        self.items.append(x)\n    def pop(self):\n        return self.items.pop()",
    "for i in range(10):\n    if i % 2 == 0:\n        print(f'Even: {i}')",
] * 10  # 50 examples

EVAL_MEDICAL_TEXTS = [
    "The patient presented with acute myocardial infarction. ECG showed ST elevation in leads II, III, and aVF.",
    "Metformin inhibits hepatic gluconeogenesis via AMPK activation, reducing fasting blood glucose.",
    "MRI revealed a 2.3cm hyperintense lesion in the right temporal lobe consistent with glioblastoma multiforme.",
    "Sepsis criteria: temperature >38C or <36C, heart rate >90bpm, respiratory rate >20, WBC >12000 or <4000.",
    "The BRCA1 mutation confers a 65-85% lifetime risk of breast cancer and 39-46% risk of ovarian cancer.",
] * 10  # 50 examples

probe_code_texts    = EVAL_CODE_TEXTS    # alias used by JSD probe cell
probe_medical_texts = EVAL_MEDICAL_TEXTS # alias used by JSD probe cell

In [34]:
# ═══════════════════════════════════════════════════════════════════════════
# CHANGE 1: Gradient Cosine Similarity Diagnostic
# ═══════════════════════════════════════════════════════════════════════════
# PURPOSE
# -------
# The thesis claims parameter entanglement is caused by *anti-aligned*
# gradients (cos < 0) from two domains accumulating in the shared LoRA.
# This cell measures that directly, before any spawning occurs.
#
# Three outcomes and what they mean for the thesis:
#   cos < 0  (anti-aligned)  → "destructive interference" framing is correct.
#                               Strongest possible evidence for the thesis claim.
#   cos ≈ 0  (orthogonal)    → No active destruction, but still capacity waste.
#                               Thesis still holds — reframe as "capacity
#                               fragmentation" rather than "interference".
#   cos > 0  (aligned)       → Domains share gradient directions — entanglement
#                               hypothesis is weak. Requires major reframing.
#
# PROTOCOL
# --------
# 1. Use the FROZEN base LoRA (no prior training) — we want the gradient
#    geometry of the *conflict* itself, not a learned response to it.
# 2. Pass a pure code batch → accumulate gradients → store as g_code.
# 3. Pass a pure medical batch → accumulate gradients → store as g_medical.
# 4. Compute cosine similarity between g_code and g_medical.
# 5. Repeat across N_GRAD_TRIALS pairs and report mean ± std.
#
# This diagnostic should be run ONCE before Phase 3, on the clean model.
# Results belong in Section 3.1 of the thesis as direct empirical support.
# ═══════════════════════════════════════════════════════════════════════════

import torch
import torch.nn.functional as F
import gc
from accelerate.hooks import remove_hook_from_module, add_hook_to_module

# ── Config ────────────────────────────────────────────────────────────────
GRAD_DIAG_LAYER   = 0       # Use Layer 0 (highest Jaccard overlap)
GRAD_DIAG_EXPERT  = 0       # Target expert
N_GRAD_TRIALS     = 20      # Number of (code_batch, medical_batch) pairs
GRAD_BATCH_SIZE   = 8       # Tokens per gradient estimate
GRAD_MAX_LEN      = 64      # Keep short — we need many passes

# ── Helper: accumulate gradients from a list of texts ─────────────────────
def accumulate_domain_gradient(hier_wrapper, texts, tokenizer,
                                n_samples=GRAD_BATCH_SIZE,
                                max_length=GRAD_MAX_LEN):
    """
    Forward + backward on `n_samples` texts from one domain.
    Returns flattened gradient vector [grad_A || grad_B] as float32 CPU tensor.
    Zeroes gradients before and after to avoid cross-contamination.
    """
    lora = hier_wrapper.base_loras[GRAD_DIAG_EXPERT]

    # Zero out any residual gradients
    if lora.A.grad is not None: lora.A.grad.zero_()
    if lora.B.grad is not None: lora.B.grad.zero_()

    total_loss = torch.tensor(0.0, device=model_phi.device, requires_grad=False)
    n_valid = 0

    for text in texts[:n_samples]:
        enc = tokenizer(
            text, return_tensors='pt',
            truncation=True, max_length=max_length, padding=False
        ).to(model_phi.device)
        if enc['input_ids'].shape[1] < 2:
            continue
        labels = enc['input_ids'].clone()
        labels[:, :-1] = enc['input_ids'][:, 1:]
        labels[:, -1]  = -100
        loss = model_phi(**enc, labels=labels).loss
        loss.backward()
        n_valid += 1

    if n_valid == 0:
        return None

    # Flatten and detach gradients — normalise by number of samples
    grad_A = lora.A.grad.detach().float().cpu() / n_valid
    grad_B = lora.B.grad.detach().float().cpu() / n_valid
    g_flat = torch.cat([grad_A.flatten(), grad_B.flatten()])

    # Zero after extraction so next call starts clean
    lora.A.grad.zero_()
    lora.B.grad.zero_()

    return g_flat


# ── Setup: fresh patch on a clean model state ─────────────────────────────
gc.collect()
torch.cuda.empty_cache()

# Force-unwrap until we hit the raw packed experts (has down_proj directly)
current = model_phi.model.layers[GRAD_DIAG_LAYER].mlp.experts
while not hasattr(current, 'down_proj'):
    orig = current.original if hasattr(current, 'original') else None
    if orig is None:
        break
    if hasattr(current, '_hf_hook'):
        h = current._hf_hook
        remove_hook_from_module(current)
        add_hook_to_module(orig, h)
    model_phi.model.layers[GRAD_DIAG_LAYER].mlp.experts = orig
    current = orig

diag_wrapper = HierarchicalPhimoeExperts(
    current, base_rank=16, lora_alpha=32
)

model_phi.gradient_checkpointing_disable()
model_phi.config.use_cache = False

model_phi.model.layers[GRAD_DIAG_LAYER].mlp.experts = diag_wrapper
if hasattr(current, '_hf_hook'):
    h = current._hf_hook
    remove_hook_from_module(current)
    add_hook_to_module(diag_wrapper, h)

# Freeze everything except the target expert's base LoRA
for p in model_phi.parameters():
    p.requires_grad = False
lora_target = diag_wrapper.base_loras[GRAD_DIAG_EXPERT]
lora_target.A.requires_grad = True
lora_target.B.requires_grad = True

print(f"Gradient cosine diagnostic — Layer {GRAD_DIAG_LAYER}, Expert {GRAD_DIAG_EXPERT}")
print(f"LoRA shape: A={lora_target.A.shape}, B={lora_target.B.shape}")
print(f"Gradient vector dim: {lora_target.A.numel() + lora_target.B.numel():,}")
print(f"Running {N_GRAD_TRIALS} trials...\n")

# ── Source texts: reuse probe texts from JSD cell ─────────────────────────
# EVAL_CODE_TEXTS and EVAL_MEDICAL_TEXTS must be defined (from Cell 17/18).
# If running this cell standalone, define them here:
# EVAL_CODE_TEXTS    = [list of code strings]
# EVAL_MEDICAL_TEXTS = [list of medical strings]

cosines = []
model_phi.train()

for trial in range(N_GRAD_TRIALS):
    # Offset into text lists so each trial sees different examples
    offset = trial * GRAD_BATCH_SIZE
    code_batch    = EVAL_CODE_TEXTS[offset % len(EVAL_CODE_TEXTS):
                                     offset % len(EVAL_CODE_TEXTS) + GRAD_BATCH_SIZE]
    medical_batch = EVAL_MEDICAL_TEXTS[offset % len(EVAL_MEDICAL_TEXTS):
                                        offset % len(EVAL_MEDICAL_TEXTS) + GRAD_BATCH_SIZE]

    g_code    = accumulate_domain_gradient(diag_wrapper, code_batch, tokenizer)
    g_medical = accumulate_domain_gradient(diag_wrapper, medical_batch, tokenizer)

    if g_code is None or g_medical is None:
        print(f"  Trial {trial+1:02d}: skipped (empty batch)")
        continue

    cos = F.cosine_similarity(g_code.unsqueeze(0),
                               g_medical.unsqueeze(0)).item()
    cosines.append(cos)

    # Print per-trial for live monitoring on Kaggle
    alignment = "ANTI-ALIGNED" if cos < -0.05 else ("ORTHOGONAL" if cos < 0.05 else "ALIGNED")
    print(f"  Trial {trial+1:02d}: cosine = {cos:+.4f}  [{alignment}]")

# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("GRADIENT COSINE SIMILARITY RESULTS")
print("=" * 55)
if cosines:
    import statistics
    mean_cos = statistics.mean(cosines)
    std_cos  = statistics.stdev(cosines) if len(cosines) > 1 else 0.0
    neg_frac = sum(1 for c in cosines if c < 0) / len(cosines)

    print(f"  Mean cosine similarity : {mean_cos:+.4f}")
    print(f"  Std dev                : {std_cos:.4f}")
    print(f"  Fraction negative      : {neg_frac:.1%}  ({sum(1 for c in cosines if c < 0)}/{len(cosines)} trials)")
    print()

    if mean_cos < -0.05:
        print("  VERDICT: ANTI-ALIGNED — thesis framing (destructive interference) confirmed.")
        print("  Report: cos(∇_D1 L_k, ∇_D2 L_k) = {:.3f} ± {:.3f}".format(mean_cos, std_cos))
        print("  Use in Section 3.1 as direct empirical support for Equation (entanglement def).")
    elif mean_cos < 0.05:
        print("  VERDICT: ORTHOGONAL — gradients don't help each other but don't destroy either.")
        print("  Reframe: capacity fragmentation rather than destructive interference.")
        print("  The method still works — the mechanism description needs adjusting.")
    else:
        print("  VERDICT: ALIGNED — domains share gradient direction. Entanglement hypothesis weak.")
        print("  Consider: are code/medical more similar than expected at this layer?")
        print("  Try deeper layers or a different expert (higher Jaccard overlap).")
print("=" * 55)

# ── Teardown: restore model to unpatched state for Phase 3 ────────────────
orig_back = diag_wrapper.original
if hasattr(diag_wrapper, '_hf_hook'):
    h = diag_wrapper._hf_hook
    remove_hook_from_module(diag_wrapper)
    add_hook_to_module(orig_back, h)
model_phi.model.layers[GRAD_DIAG_LAYER].mlp.experts = orig_back
gc.collect()
torch.cuda.empty_cache()
print("\nModel restored to unpatched state. Ready for Phase 3.")


Gradient cosine diagnostic — Layer 0, Expert 0
LoRA shape: A=torch.Size([16, 4096]), B=torch.Size([4096, 16])
Gradient vector dim: 131,072
Running 20 trials...

  Trial 01: cosine = +0.0050  [ORTHOGONAL]
  Trial 02: cosine = -0.0064  [ORTHOGONAL]
  Trial 03: cosine = -0.0029  [ORTHOGONAL]
  Trial 04: cosine = -0.0025  [ORTHOGONAL]
  Trial 05: cosine = -0.0013  [ORTHOGONAL]
  Trial 06: cosine = -0.0065  [ORTHOGONAL]
  Trial 07: cosine = -0.0057  [ORTHOGONAL]
  Trial 08: cosine = +0.0015  [ORTHOGONAL]
  Trial 09: cosine = -0.0045  [ORTHOGONAL]
  Trial 10: cosine = +0.0014  [ORTHOGONAL]
  Trial 11: cosine = +0.0007  [ORTHOGONAL]
  Trial 12: cosine = -0.0072  [ORTHOGONAL]
  Trial 13: cosine = +0.0012  [ORTHOGONAL]
  Trial 14: cosine = -0.0033  [ORTHOGONAL]
  Trial 15: cosine = -0.0000  [ORTHOGONAL]
  Trial 16: cosine = -0.0010  [ORTHOGONAL]
  Trial 17: cosine = -0.0066  [ORTHOGONAL]
  Trial 18: cosine = -0.0024  [ORTHOGONAL]
  Trial 19: cosine = -0.0047  [ORTHOGONAL]
  Trial 20: cosine = -

## Section 7 — Phase 2: Hyperparameter Calibration Grid

In [ ]:
# ── Trigger Diagnostic: why is nothing firing? ────────────────────────────────
# Run ONE trial manually and log the two trigger signals every step
# so we can see whether plateau, high_error, or both are failing.

import torch
gc.collect(); torch.cuda.empty_cache()

# Fresh patch
current = model_phi.model.layers[TARGET_LAYER].mlp.experts
if isinstance(current, HierarchicalPhimoeExperts):
    original_experts = current.original
    if hasattr(current, '_hf_hook'):
        h = current._hf_hook
        remove_hook_from_module(current); add_hook_to_module(original_experts, h)
    model_phi.model.layers[TARGET_LAYER].mlp.experts = original_experts
else:
    original_experts = current

hier = HierarchicalPhimoeExperts(original_experts, base_rank=BASE_RANK, lora_alpha=BASE_RANK*2)
model_phi.model.layers[TARGET_LAYER].mlp.experts = hier
if hasattr(original_experts, '_hf_hook'):
    h = original_experts._hf_hook
    remove_hook_from_module(original_experts); add_hook_to_module(hier, h)

optimizer = torch.optim.AdamW(
    [p for p in hier.parameters() if p.requires_grad], lr=LR)

# Use aggressive settings to maximise chance of firing
test_alpha       = 1.05
test_tau_plateau = 1e-4
monitor = SaturationMonitor(expert_id=TARGET_EXPERT, window=10,
                             tau_plateau=test_tau_plateau, alpha=test_alpha)

print(f"Diagnostic run: alpha={test_alpha}, tau_plateau={test_tau_plateau}")
print(f"{'Step':>5} {'domain':>8} {'loss':>8} {'exp_proxy':>10} "
      f"{'ri_slope':>12} {'plateau':>8} {'high_err':>9}")
print("-" * 70)

ri_history = []
ema_decay  = 0.9
ema_expert = None
ema_batch  = None

for step in range(100):
    idx       = step % len(calib_domains)
    input_ids = calib_encodings['input_ids'][idx].unsqueeze(0)
    labels    = input_ids.clone()
    labels[:, :-1] = input_ids[:, 1:]
    labels[:, -1]  = -100

    optimizer.zero_grad()
    loss = model_phi(input_ids=input_ids, labels=labels).loss
    loss.backward()

    lora = hier.base_loras[TARGET_EXPERT]
    is_medical = (calib_domains[idx] == "medical")
    expert_proxy = loss.item() * (1.15 if is_medical else 0.90)

    # Manually compute what the monitor sees
    if lora.A.grad is not None and lora.B.grad is not None:
        # Rank importance: mean(||B[:,r]|| * ||A[r,:]||)
        col_norms = lora.B.data.detach().float().norm(dim=0)
        row_norms = lora.A.data.detach().float().norm(dim=1)
        ri = (col_norms * row_norms).mean().item()
    else:
        ri = 0.0
    ri_history.append(ri)

    # OLS slope over last 10 steps
    if len(ri_history) >= 10:
        recent = ri_history[-10:]
        x = torch.arange(10, dtype=torch.float32)
        y = torch.tensor(recent, dtype=torch.float32)
        slope = ((x*y).mean() - x.mean()*y.mean()) / (x.var(unbiased=False) + 1e-12)
        slope_val = slope.item()
    else:
        slope_val = float('nan')

    # EMA tracking
    d = ema_decay
    ema_expert = expert_proxy if ema_expert is None else d*ema_expert + (1-d)*expert_proxy
    ema_batch  = loss.item()  if ema_batch  is None else d*ema_batch  + (1-d)*loss.item()

    plateau   = abs(slope_val) < test_tau_plateau if not torch.isnan(torch.tensor(slope_val)) else False
    high_err  = ema_expert > test_alpha * ema_batch

    if step % 5 == 0:
        domain_str = "medical" if is_medical else "code"
        print(f"{step:>5} {domain_str:>8} {loss.item():>8.4f} {expert_proxy:>10.4f} "
              f"{slope_val:>12.2e} {str(plateau):>8} {str(high_err):>9}")

    optimizer.step()

print("\n── Key values at final step ──")
print(f"  ema_expert = {ema_expert:.4f}")
print(f"  ema_batch  = {ema_batch:.4f}")
print(f"  ratio      = {ema_expert/ema_batch:.4f}  (need > {test_alpha})")
print(f"  ri range   = [{min(ri_history):.2e}, {max(ri_history):.2e}]")
print(f"  slope range ≈ last value = {slope_val:.2e}  (need < {test_tau_plateau})")

In [ ]:
# ── Phase 2: Hyperparameter Calibration Grid ─────────────────────────────────
# Goal: find (alpha, tau_plateau) that gives 1-3 spawns per 100 steps.
# 9 combinations × 300 steps each. Uses existing HierarchicalPhimoeExperts
# and SaturationMonitor — no architecture changes from Stage 2.

import gc, torch, itertools
from accelerate.hooks import remove_hook_from_module, add_hook_to_module

calib_results = []   # will hold dicts with settings + spawn count

# ── Updated calibration grid ──────────────────────────────────────────────────
DELTA_VALUES      = [0.3, 0.5, 1.0]    # |ema_medical - ema_code| threshold
TAU_PLATEAU_VALUES = [1e-3, 5e-4, 1e-4]

def run_calibration_trial_v2(delta_threshold, tau_plateau,
                              encodings, domains, n_steps=CALIB_STEPS):
    gc.collect(); torch.cuda.empty_cache()

    # Fresh patch (same unwrap logic as before)
    current = model_phi.model.layers[TARGET_LAYER].mlp.experts
    if isinstance(current, HierarchicalPhimoeExperts):
        original_experts = current.original
        if hasattr(current, '_hf_hook'):
            h = current._hf_hook
            remove_hook_from_module(current); add_hook_to_module(original_experts, h)
        model_phi.model.layers[TARGET_LAYER].mlp.experts = original_experts
    else:
        original_experts = current

    hier = HierarchicalPhimoeExperts(original_experts, base_rank=BASE_RANK,
                                      lora_alpha=BASE_RANK*2)
    model_phi.model.layers[TARGET_LAYER].mlp.experts = hier
    if hasattr(original_experts, '_hf_hook'):
        h = original_experts._hf_hook
        remove_hook_from_module(original_experts); add_hook_to_module(hier, h)

    monitor = ConflictSaturationMonitor(
        tau_plateau=tau_plateau,
        delta_threshold=delta_threshold,
        window=15,
    )
    optimizer = torch.optim.AdamW(
        [p for p in hier.parameters() if p.requires_grad], lr=LR)

    spawn_steps, loss_log = [], []

    for step in range(n_steps):
        idx       = step % len(domains)
        input_ids = encodings['input_ids'][idx].unsqueeze(0)
        labels    = input_ids.clone()
        labels[:, :-1] = input_ids[:, 1:]
        labels[:, -1]  = -100

        optimizer.zero_grad()
        loss = model_phi(input_ids=input_ids, labels=labels).loss
        loss_log.append(loss.item())
        loss.backward()

        lora   = hier.base_loras[TARGET_EXPERT]
        domain = domains[idx]

        triggered = monitor.update(
            lora_A=lora.A.data,
            lora_B=lora.B.data,
            loss_val=loss.item(),
            domain=domain,
        )

        if triggered:
            grad_A, grad_B = lora.A.grad, lora.B.grad
            weight_grad = (grad_B.detach().float() @ grad_A.detach().float()
                           if grad_A is not None and grad_B is not None else None)
            new_params = hier.spawn(TARGET_EXPERT, rank=8, weight_grad=weight_grad)
            optimizer.add_param_group({'params': new_params, 'lr': LR})
            monitor.reset_after_spawn()
            spawn_steps.append(step)

        optimizer.step()

    spawn_rate = len(spawn_steps) / (n_steps / 100)
    return {
        "delta_threshold": delta_threshold,
        "tau_plateau":     tau_plateau,
        "spawns":          len(spawn_steps),
        "spawn_steps":     spawn_steps,
        "spawn_rate":      spawn_rate,
        "final_loss":      loss_log[-1],
    }


# Run updated grid
print("=" * 65)
print("PHASE 2 (v2): CALIBRATION WITH CONFLICT DELTA TRIGGER")
print(f"Grid: delta={DELTA_VALUES} × tau_plateau={TAU_PLATEAU_VALUES}")
print("=" * 65)

calib_results_v2 = []
for delta, tau in itertools.product(DELTA_VALUES, TAU_PLATEAU_VALUES):
    print(f"\n── delta={delta}  tau_plateau={tau:.0e} ──")
    r = run_calibration_trial_v2(delta, tau, calib_encodings, calib_domains)
    calib_results_v2.append(r)
    tag = ("✅ GOOD" if 1 <= r["spawn_rate"] <= 3
           else "🔥 TOO MANY" if r["spawn_rate"] > 3 else "❌ NONE")
    print(f"   Spawns: {r['spawns']}  |  Rate: {r['spawn_rate']:.1f}/100 steps  |  {tag}")

print("\n" + "=" * 65)
print("SUMMARY")
print(f"{'delta':>8}  {'tau_plateau':>12}  {'spawns':>7}  {'rate/100':>9}  verdict")
print("-" * 65)
good = []
for r in calib_results_v2:
    rate = r["spawn_rate"]
    tag  = "✅ GOOD" if 1 <= rate <= 3 else ("🔥 TOO MANY" if rate > 3 else "❌ NONE")
    print(f"{r['delta_threshold']:>8.1f}  {r['tau_plateau']:>12.0e}  "
          f"{r['spawns']:>7}  {rate:>9.1f}  {tag}")
    if 1 <= rate <= 3:
        good.append(r)

print("=" * 65)
if good:
    best = min(good, key=lambda r: r["spawn_rate"])
    print(f"\n✅ LOCKED HYPERPARAMETERS:")
    print(f"   delta_threshold = {best['delta_threshold']}")
    print(f"   tau_plateau     = {best['tau_plateau']:.0e}")
    LOCKED_DELTA       = best["delta_threshold"]
    LOCKED_TAU_PLATEAU = best["tau_plateau"]
else:
    print("\n⚠️  Still no spawns. Check that loss variance between domains is >0.3")
    print("   Print monitor._ema_code and monitor._ema_medical mid-run to verify.")

## Section 8 — JSD Sub-Router Specialisation Probe

In [35]:
# ── Phase 1 / Phase 4: JSD Sub-Router Specialisation Probe ───────────────────
# Measures Jensen-Shannon Divergence between ReLU gate activation distributions
# P_code and P_medical on each spawned sub-adapter.
#
# High JSD (→1.0) = gates have partitioned the two domains → entanglement resolved
# Low JSD  (→0.0) = gates treat both domains identically → no specialisation
#
# Call compute_jsd_probe() at checkpoints during Phase 3 training.

import torch
import torch.nn.functional as F
import numpy as np
from collections import defaultdict

def collect_gate_activations(hier_experts, tokenizer, texts, domain_label,
                              max_length=128, n_examples=50):
    activations = defaultdict(list)
    model_phi.eval()

    with torch.no_grad():
        for text in texts[:n_examples]:
            inputs = tokenizer(
                text, return_tensors='pt',
                truncation=True, max_length=max_length, padding=False
            ).to(hier_experts.device)

            captured = {}
            def hook_fn(module, input, output):
                h = input[0].detach()
                # FIX: flatten batch/seq dimensions → always [T, model_dim]
                if h.dim() == 3:
                    h = h.view(-1, h.shape[-1])
                captured['hidden'] = h

            hook = model_phi.model.layers[TARGET_LAYER].mlp.register_forward_hook(hook_fn)
            _ = model_phi(**inputs)
            hook.remove()

            if 'hidden' not in captured:
                continue

            hidden = captured['hidden']  # now guaranteed [T, model_dim]

            for j, gate_vec in enumerate(hier_experts.spawn_gates[TARGET_EXPERT]):
                gate_acts = F.relu(hidden @ gate_vec.to(hidden.dtype))  # [T]
                activations[j].extend(gate_acts.float().cpu().tolist())

    model_phi.train()
    return activations


def compute_jsd(p_vals, q_vals, n_bins=50):
    """
    Compute Jensen-Shannon Divergence between two empirical distributions.
    Bins both into a shared histogram, then computes JSD.
    Returns scalar in [0, 1] (using log base 2, so max=1).
    """
    if len(p_vals) < 5 or len(q_vals) < 5:
        return float('nan')

    # Shared bin range
    all_vals = p_vals + q_vals
    min_v, max_v = min(all_vals), max(all_vals)
    if max_v - min_v < 1e-8:
        return 0.0   # both distributions are identical point masses

    bins = np.linspace(min_v, max_v, n_bins + 1)
    p_hist, _ = np.histogram(p_vals, bins=bins, density=False)
    q_hist, _ = np.histogram(q_vals, bins=bins, density=False)

    # Normalise to probability distributions with Laplace smoothing
    p = (p_hist + 1e-8) / (p_hist.sum() + 1e-8 * n_bins)
    q = (q_hist + 1e-8) / (q_hist.sum() + 1e-8 * n_bins)

    m = 0.5 * (p + q)
    # JSD = 0.5 * KL(P||M) + 0.5 * KL(Q||M), base-2
    jsd = 0.5 * np.sum(p * np.log2(p / m + 1e-12)) + \
          0.5 * np.sum(q * np.log2(q / m + 1e-12))
    return float(np.clip(jsd, 0.0, 1.0))


def compute_jsd_probe(hier_experts, tokenizer,
                       code_texts, medical_texts,
                       step=None, n_examples=50, verbose=True):
    """
    Main probe function. Call this at training checkpoints.

    Returns dict: {sub_adapter_idx: jsd_value}
    A JSD > 0.3 is meaningful specialisation.
    A JSD > 0.6 is strong evidence of domain partitioning.
    """
    n_spawned = len(hier_experts.spawn_gates[TARGET_EXPERT])
    if n_spawned == 0:
        if verbose:
            print("  [JSD probe] No sub-adapters spawned yet — skipping.")
        return {}

    if verbose:
        label = f" (step {step})" if step is not None else ""
        print(f"\n── JSD Probe{label} ──  "
              f"{n_spawned} sub-adapter(s) on L{TARGET_LAYER} E{TARGET_EXPERT}")

    code_acts    = collect_gate_activations(
        hier_experts, tokenizer, code_texts,    "code",    n_examples=n_examples)
    medical_acts = collect_gate_activations(
        hier_experts, tokenizer, medical_texts, "medical", n_examples=n_examples)

    jsd_results = {}
    for j in range(n_spawned):
        p_code    = code_acts.get(j, [])
        p_medical = medical_acts.get(j, [])
        jsd       = compute_jsd(p_code, p_medical)
        jsd_results[j] = jsd

        if verbose:
            bar_len = int(jsd * 20) if not np.isnan(jsd) else 0
            bar     = "█" * bar_len + "░" * (20 - bar_len)
            verdict = ("🟢 strong" if jsd > 0.6
                       else "🟡 moderate" if jsd > 0.3
                       else "🔴 weak" if not np.isnan(jsd)
                       else "⚪ n/a")
            print(f"  Sub-adapter {j}: JSD={jsd:.3f}  [{bar}]  {verdict}")

    return jsd_results


# ── Build held-out probe texts (no overlap with training data) ────────────────
# Use the LAST n examples from each dataset — training uses the first N_EACH
# ── Build held-out probe texts ────────────────────────────────────────────────
print("Building held-out probe texts...")

PROBE_N = 50

primary_name,  primary_col,  primary_cfg  = DATASET_CFG["primary"]
conflict_name, conflict_col, conflict_cfg = DATASET_CFG["conflict"]

_primary_full  = load_split_auto(primary_name,  primary_cfg)
_conflict_full = load_split_auto(conflict_name, conflict_cfg)

# HumanEval only has 164 examples and was never used for training
# (training used calibration encodings, not this dataset directly),
# so the full split is clean to use as probe.
probe_code_texts = [
    str(ex[primary_col])
    for ex in _primary_full.select(range(min(PROBE_N, len(_primary_full))))
]

# PubMedQA has 1000 examples — skip first N_EACH to avoid train overlap
_conflict_skip = min(N_EACH, len(_conflict_full) - PROBE_N)
probe_medical_texts = [
    str(ex[conflict_col])
    for ex in _conflict_full.select(
        range(_conflict_skip, min(_conflict_skip + PROBE_N, len(_conflict_full)))
    )
]

print(f"Probe texts: {len(probe_code_texts)} code, "
      f"{len(probe_medical_texts)} medical (held-out)")

# FIX: use whatever the current patched experts variable is called.
# Grab it directly from the model so this cell works regardless of
# what the calibration cell named its local variable.
# Ensure we're probing a freshly patched, untrained wrapper
# (calibration left a trained hier in the layer — reset it)
current = model_phi.model.layers[TARGET_LAYER].mlp.experts
if isinstance(current, HierarchicalPhimoeExperts):
    original_experts = current.original
    if hasattr(current, '_hf_hook'):
        h = current._hf_hook
        remove_hook_from_module(current)
        add_hook_to_module(original_experts, h)
    model_phi.model.layers[TARGET_LAYER].mlp.experts = original_experts

# Fresh patch for probing
original_experts = model_phi.model.layers[TARGET_LAYER].mlp.experts
hier_experts = HierarchicalPhimoeExperts(
    original_experts, base_rank=BASE_RANK, lora_alpha=BASE_RANK*2)
model_phi.model.layers[TARGET_LAYER].mlp.experts = hier_experts
if hasattr(original_experts, '_hf_hook'):
    h = original_experts._hf_hook
    remove_hook_from_module(original_experts)
    add_hook_to_module(hier_experts, h)

print(f"Fresh patch applied. Sub-adapters: "
      f"{len(hier_experts.spawn_gates[TARGET_EXPERT])}")  # should be 0

print("\nTest run on current model state:")
_ = compute_jsd_probe(hier_experts, tokenizer,
                       probe_code_texts, probe_medical_texts, step=0)

Building held-out probe texts...
    [openai/openai_humaneval] using split='test' (available: ['test'])
    [qiaojin/PubMedQA] using split='train' (available: ['train'])
Probe texts: 50 code, 50 medical (held-out)
Fresh patch applied. Sub-adapters: 0

Test run on current model state:
  [JSD probe] No sub-adapters spawned yet — skipping.


## Section 9 — Phase 3: Conflict-Scaling Grid

In [51]:
# ── Phase 3 Config ─────────────────────────────────────────────────────────────
# All settings in one place. Do not change after grid starts.

PHASE3_CFG = {
    # Locked from Phase 2
    "delta_threshold": 1.0,
    "tau_plateau":     1e-4,
    "target_layer":    0,
    "target_expert":   0,
    "base_rank":       16,
    "lr":              2e-5,
    "sigma_scale":     1e-3,

    # Training
    "n_steps":         1500,     # per run — enough to see negative transfer emerge
    "max_sub_adapters": 10,     # cap on spawns per expert
    "eval_every":      100,     # steps between JSD + perplexity checkpoints
    "max_length":      128,
    "batch_size":      1,

    # Grid
    "conflict_ratios": {
        "A": 0.0,   # 100% code
        "B": 0.2,   # 80/20
        "C": 0.5,   # 50/50
    },
    "methods": ["lora", "dr_lora", "hierarchical"],
}

# Evaluation: held-out perplexity as proxy for Python accuracy
# (avoids needing code execution infrastructure)
# Lower perplexity on code probe = less negative transfer
EVAL_CODE_TEXTS    = probe_code_texts      # from JSD probe cell
EVAL_MEDICAL_TEXTS = probe_medical_texts

print("Phase 3 config locked.")
print(f"Total runs: {len(PHASE3_CFG['methods'])} methods × "
      f"{len(PHASE3_CFG['conflict_ratios'])} ratios = "
      f"{len(PHASE3_CFG['methods']) * len(PHASE3_CFG['conflict_ratios'])} training runs")
print(f"Steps per run: {PHASE3_CFG['n_steps']}")

Phase 3 config locked.
Total runs: 3 methods × 3 ratios = 9 training runs
Steps per run: 1500


In [44]:
# ── PackedDRLoRAExperts ───────────────────────────────────────────────────────
# Full DR-LoRA implementation for Phi-MoE's packed expert format.
#
# WHY THIS EXISTS
# ───────────────
# apply_dr_lora_to_moe_experts (teammate's code) assumes experts are a Python
# list of individual nn.Module objects with named nn.Linear children — the
# Mixtral/Mistral layout. Phi-MoE stores all expert weights as a single packed
# tensor: down_proj ∈ R^[num_experts, model_dim, ffn_dim]. There are no
# individual expert objects to iterate or setattr on.
#
# PackedDRLoRAExperts wraps the entire PhimoeExperts block and attaches one
# DRLoRALayer per expert, operating in model_dim space (same as
# HierarchicalPhimoeExperts). It exposes a get_lora_modules_list() method
# that returns a list in the exact format DRLoRATracker.compute_rank_importance
# and perform_rank_growth expect, so the full saliency pipeline works unchanged.
#
# FORWARD PASS
# ────────────
# E_k(x) = [original packed expert k output] + DRLoRALayer_k(x) * eff_weight_k
#
# Only tokens routed to expert k (eff_weight > threshold) pass through
# DRLoRALayer_k. The correction is weighted by the router's softmax weight for
# that expert, exactly mirroring how the base expert output is weighted.
#
# RANK GROWTH
# ───────────
# Each DRLoRALayer has a binary rank_mask. perform_rank_growth calls
# activate_rank(j) on the DRLoRALayer directly via the lora_modules_list.
# No tensor resizing — capacity reservation is identical to teammate's design.

import torch
import torch.nn as nn
from accelerate.hooks import remove_hook_from_module, add_hook_to_module


class PackedDRLoRAExperts(nn.Module):
    """
    Drop-in replacement for PhimoeExperts that adds per-expert DRLoRALayer
    adapters compatible with DRLoRATracker and perform_rank_growth.
    """

    def __init__(self, original_experts, r_max=16, r_init=4, lora_alpha=16):
        super().__init__()
        self.original = original_experts

        # Freeze all original expert parameters
        for p in self.original.parameters():
            p.requires_grad = False

        # Infer dimensions from packed weight
        # down_proj: [num_experts, model_dim, ffn_dim]
        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype
        dev = self.original.down_proj.device

        self.r_max  = r_max
        self.r_init = r_init

        # One DRLoRALayer per expert, operating in model_dim → model_dim space.
        # lora_dim = out_f = model_dim (same reasoning as HierarchicalPhimoeExperts).
        self.dr_loras = nn.ModuleList([
            DRLoRALayer(
                in_features=self.out_f,
                out_features=self.out_f,
                r_max=r_max,
                r_init=r_init,
                lora_alpha=lora_alpha,
                lora_dropout=0.0,
            ).to(device=dev, dtype=self.dtype)
            for _ in range(self.num_experts)
        ])

    # ── Compatibility properties ──────────────────────────────────────────────

    @property
    def device(self):
        return self.original.down_proj.device

    # ── lora_modules_list for DRLoRATracker / perform_rank_growth ─────────────

    def get_lora_modules_list(self, layer_idx):
        """
        Returns a list of dicts in the format expected by:
          - DRLoRATracker.compute_rank_importance(lora_modules)
          - perform_rank_growth(lora_modules=...)

        Each dict has keys: 'layer', 'expert', 'dr_lora'.
        'dr_lora' is the DRLoRALayer for that expert — tracker reads
        lora_A / lora_B and their gradients directly from it.
        """
        return [
            {
                "layer":   layer_idx,
                "expert":  k,
                "dr_lora": self.dr_loras[k],
                # tracker also reads in_features/out_features for param counts
                "module":  "packed_lora",
            }
            for k in range(self.num_experts)
        ]

    # ── Forward ───────────────────────────────────────────────────────────────

    def forward(self, hidden_states, top_k_indices, top_k_weights):
        """
        hidden_states : [T, model_dim]
        top_k_indices : [T, top_k]   — which experts each token routes to
        top_k_weights : [T, num_experts] — full softmax weight vector

        Returns [T, model_dim].
        """
        # Original packed expert output (frozen)
        orig_out   = self.original.forward(hidden_states, top_k_indices, top_k_weights)
        correction = torch.zeros_like(orig_out)

        for k in range(self.num_experts):
            # Routing weight for expert k per token from full softmax vector
            eff_w = top_k_weights[:, k]          # [T]

            # Only compute correction for tokens that actually route to expert k
            # (weight above numerical threshold — avoids grad on truly-zero paths)
            active = eff_w.abs() > 1e-6          # [T] bool
            if not active.any():
                continue

            x_k   = hidden_states[active]         # [T_active, model_dim]
            lora_k = self.dr_loras[k]

            # Cast to match hidden_states dtype (bf16 in practice)
            lora_out = lora_k(x_k)               # [T_active, model_dim]

            # Weight the correction by the router's softmax weight
            correction[active] += eff_w[active].unsqueeze(1) * lora_out

        return orig_out + correction


print("✅ PackedDRLoRAExperts defined")

✅ PackedDRLoRAExperts defined


In [45]:
# ── Method Factory ─────────────────────────────────────────────────────────────
# Returns (patched_model_component, optimizer, monitor_or_none) for each method.

import torch.nn as nn
from accelerate.hooks import remove_hook_from_module, add_hook_to_module


def get_clean_original_experts(target_layer):
    """
    Unwraps any wrapper and returns raw PhimoeExperts (has down_proj directly).
    Uses down_proj presence as ground truth — more robust than isinstance checks.
    """
    current = model_phi.model.layers[target_layer].mlp.experts
    while not hasattr(current, 'down_proj'):
        orig = getattr(current, 'original', None)
        if orig is None:
            break
        if hasattr(current, '_hf_hook'):
            h = current._hf_hook
            remove_hook_from_module(current)
            add_hook_to_module(orig, h)
        model_phi.model.layers[target_layer].mlp.experts = orig
        current = orig
    return current


def _patch_layer(wrapper, original_experts, target_layer):
    """Install wrapper and transfer accelerate hook."""
    model_phi.model.layers[target_layer].mlp.experts = wrapper
    if hasattr(original_experts, '_hf_hook'):
        h = original_experts._hf_hook
        remove_hook_from_module(original_experts)
        add_hook_to_module(wrapper, h)


class RealDRLoRAWrapper:
    """
    Thin container so the grid runner can treat DR-LoRA uniformly.
    Holds the PackedDRLoRAExperts wrapper + tracker + schedule.
    """
    def __init__(self, packed_wrapper, tracker, schedule, layer_idx):
        self.packed_wrapper = packed_wrapper
        self.tracker        = tracker
        self.schedule       = schedule
        self.layer_idx      = layer_idx
        self.lora_modules   = packed_wrapper.get_lora_modules_list(layer_idx)
        self.expanded       = False   # flips True on first growth event

    # ── Proxy attributes so evaluate_checkpoint works unchanged ───────────────
    # evaluate_checkpoint calls wrapper.spawn_gates[expert] for JSD on hierarchical
    # and wrapper.expanded for dr_lora — both handled via method_name check there.


def setup_method(method_name, cfg):
    """
    Returns (wrapper, optimizer, monitor_or_none).
    Always starts from a clean unpatched layer.
    """
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    target_layer  = cfg["target_layer"]
    target_expert = cfg["target_expert"]
    base_rank     = cfg["base_rank"]
    lr            = cfg["lr"]
    n_steps       = cfg["n_steps"]

    original_experts = get_clean_original_experts(target_layer)

    # ── Standard LoRA (spawning disabled) ─────────────────────────────────────
    if method_name == "lora":
        wrapper = HierarchicalPhimoeExperts(
            original_experts, base_rank=base_rank, lora_alpha=base_rank * 2
        )
        _patch_layer(wrapper, original_experts, target_layer)
        optimizer = torch.optim.AdamW(
            [p for p in wrapper.parameters() if p.requires_grad], lr=lr
        )
        return wrapper, optimizer, None

    # ── Hierarchical Spawning (full method) ───────────────────────────────────
    elif method_name == "hierarchical":
        wrapper = HierarchicalPhimoeExperts(
            original_experts, base_rank=base_rank, lora_alpha=base_rank * 2
        )
        _patch_layer(wrapper, original_experts, target_layer)
        monitor = ConflictSaturationMonitor(
            tau_plateau=cfg["tau_plateau"],
            delta_threshold=cfg["delta_threshold"],
            window=15,
        )
        optimizer = torch.optim.AdamW(
            [p for p in wrapper.parameters() if p.requires_grad], lr=lr
        )
        return wrapper, optimizer, monitor

    # ── Full DR-LoRA with saliency-based growth (teammate's pipeline) ──────────
    elif method_name == "dr_lora":
        # PackedDRLoRAExperts handles Phi-MoE's packed weight format and exposes
        # get_lora_modules_list() so DRLoRATracker / perform_rank_growth work
        # without any changes to the teammate's code.
        for p in model_phi.parameters():
            p.requires_grad = False

        packed_wrapper = PackedDRLoRAExperts(
            original_experts,
            r_max=base_rank,
            r_init=base_rank // 4,   # starts at rank 4, grows to 16
            lora_alpha=base_rank,
        )
        _patch_layer(packed_wrapper, original_experts, target_layer)

        # Growth schedule — calibrated to n_steps
        warmup   = max(10, n_steps // 10)
        interval = max(10, n_steps // 8)
        end_buf  = max(10, n_steps // 10)

        dr_schedule = DRLoRAGrowthSchedule(
            total_steps=n_steps,
            warmup_steps=warmup,
            growth_interval=interval,
            r_init=base_rank // 4,
            r_target=base_rank,
            r_max=base_rank,
            num_experts_per_layer=packed_wrapper.num_experts,
            num_layers=1,
            end_buffer_steps=end_buf,
        )

        # Tracker: num_layers = full model depth so hook indices align correctly
        dr_tracker = DRLoRATracker(
            num_layers=model_phi.config.num_hidden_layers,
            num_experts_per_layer=packed_wrapper.num_experts,
            ema_beta=0.9,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
        dr_tracker.register_routing_hooks(model_phi)

        lora_params = [p for p in packed_wrapper.parameters() if p.requires_grad]
        optimizer   = torch.optim.AdamW(lora_params, lr=lr)

        wrapper = RealDRLoRAWrapper(packed_wrapper, dr_tracker, dr_schedule, target_layer)
        return wrapper, optimizer, None

    else:
        raise ValueError(f"Unknown method: {method_name}")


print("✅ Method factory ready (lora / hierarchical / dr_lora)")

✅ Method factory ready (lora / hierarchical / dr_lora)


In [38]:
# ── Evaluation Helper ──────────────────────────────────────────────────────────
# Measures perplexity on held-out code and medical texts.
# Perplexity increase on code (relative to Run A ceiling) = negative transfer.

def evaluate_perplexity(texts, tokenizer, max_length=128, n_eval=30):
    """
    Returns mean perplexity over texts. Lower = better retention of domain.
    """
    model_phi.eval()
    losses = []
    with torch.no_grad():
        for text in texts[:n_eval]:
            inputs = tokenizer(
                text, return_tensors='pt', truncation=True,
                max_length=max_length, padding=False
            ).to(model_phi.device)
            if inputs['input_ids'].shape[1] < 2:
                continue
            labels          = inputs['input_ids'].clone()
            labels[:, :-1]  = inputs['input_ids'][:, 1:]
            labels[:, -1]   = -100
            loss = model_phi(**inputs, labels=labels).loss
            losses.append(loss.item())
    model_phi.train()
    if not losses:
        return float('nan')
    mean_loss = sum(losses) / len(losses)
    return float(torch.exp(torch.tensor(mean_loss)).item())  # perplexity


def evaluate_checkpoint(wrapper, method_name, run_label, step, cfg):
    """
    Run full evaluation at a checkpoint. Returns dict of metrics.
    """
    metrics = {"method": method_name, "run": run_label, "step": step}

    # Perplexity on both domains
    metrics["ppl_code"]    = evaluate_perplexity(
        EVAL_CODE_TEXTS, tokenizer, cfg["max_length"])
    metrics["ppl_medical"] = evaluate_perplexity(
        EVAL_MEDICAL_TEXTS, tokenizer, cfg["max_length"])

    # JSD (only meaningful if sub-adapters exist)
    jsd_scores = {}
    if method_name == "hierarchical":
        jsd_scores = compute_jsd_probe(
            wrapper, tokenizer,
            EVAL_CODE_TEXTS, EVAL_MEDICAL_TEXTS,
            step=step, verbose=False
        )
    metrics["jsd_scores"] = jsd_scores
    metrics["mean_jsd"]   = (sum(jsd_scores.values()) / len(jsd_scores)
                              if jsd_scores else float('nan'))

    n_spawns = (len(wrapper.spawn_gates[cfg["target_expert"]])
                if method_name == "hierarchical" else
                (1 if (method_name == "dr_lora" and wrapper.expanded) else 0))
    metrics["n_expansions"] = n_spawns

    print(f"  [{method_name} | Run {run_label} | step {step}]  "
          f"ppl_code={metrics['ppl_code']:.2f}  "
          f"ppl_medical={metrics['ppl_medical']:.2f}  "
          f"mean_jsd={metrics['mean_jsd']:.3f}  "
          f"expansions={n_spawns}")
    return metrics

print("Evaluation helper ready.")

Evaluation helper ready.


In [52]:
# ── Phase 3 Grid Runner ────────────────────────────────────────────────────────
import gc, itertools

all_results  = []
all_run_logs = {}

model_phi.config.use_cache = False

for method_name, (run_label, conflict_ratio) in itertools.product(
        PHASE3_CFG["methods"], PHASE3_CFG["conflict_ratios"].items()):

    run_key = (method_name, run_label)
    print("\n" + "=" * 65)
    print(f"METHOD: {method_name.upper()}  |  Run {run_label} "
          f"(conflict={conflict_ratio:.0%})")
    print("=" * 65)

    gc.collect(); torch.cuda.empty_cache()

    enc, domains = build_calibration_dataloader(
        tokenizer,
        conflict_ratio=conflict_ratio,
        max_length=PHASE3_CFG["max_length"],
        n_each=PHASE3_CFG["n_steps"] + 50,
    )

    wrapper, optimizer, monitor = setup_method(method_name, PHASE3_CFG)

    loss_log            = []
    spawn_log           = []
    dr_routers_unfrozen = False
    dr_existing_ids     = set(id(p) for pg in optimizer.param_groups for p in pg["params"])

    for step in range(PHASE3_CFG["n_steps"]):
        idx       = step % len(domains)
        input_ids = enc["input_ids"][idx].unsqueeze(0)
        labels    = input_ids.clone()
        labels[:, :-1] = input_ids[:, 1:]
        labels[:, -1]  = -100

        optimizer.zero_grad()
        loss = model_phi(input_ids=input_ids, labels=labels).loss
        loss_log.append(loss.item())
        loss.backward()

        domain = domains[idx]

        # ── Method-specific post-backward ──────────────────────────────────────
        if method_name == "hierarchical":
            lora = wrapper.base_loras[PHASE3_CFG["target_expert"]]
            if lora.A.grad is not None and lora.B.grad is not None:
                triggered = monitor.update(
                    lora_A=lora.A.data, lora_B=lora.B.data,
                    loss_val=loss.item(), domain=domain,
                )
                n_spawned = len(wrapper.spawn_gates[PHASE3_CFG["target_expert"]])
                if triggered and n_spawned < PHASE3_CFG["max_sub_adapters"]:
                    wg = (lora.B.grad.detach().float() @ lora.A.grad.detach().float()
                          if lora.A.grad is not None else None)
                    new_params = wrapper.spawn(
                        PHASE3_CFG["target_expert"], rank=8, weight_grad=wg)
                    optimizer.add_param_group(
                        {"params": new_params, "lr": PHASE3_CFG["lr"]})
                    monitor.reset_after_spawn()
                    spawn_log.append(step)

        elif method_name == "dr_lora":
            # ── C: Routing frequency EMA (Eq. 5) ─────────────────────────────
            wrapper.tracker.update_routing_frequency_from_hooks()

            # ── D: Rank importance EMA (Eq. 6-8) ─────────────────────────────
            # lora_modules list points to the PackedDRLoRAExperts dr_loras —
            # tracker reads .lora_A / .lora_B and their .grad directly
            imp = wrapper.tracker.compute_rank_importance(wrapper.lora_modules)
            wrapper.tracker.update_rank_importance(imp)

            # ── E: Router unfreeze at warmup (Algorithm 1, line 5-7) ──────────
            warmup_step = max(10, PHASE3_CFG["n_steps"] // 10)
            if step == warmup_step and not dr_routers_unfrozen:
                unfreeze_routers(model_phi)
                new_rp = [p for p in model_phi.parameters()
                           if p.requires_grad and id(p) not in dr_existing_ids]
                if new_rp:
                    optimizer.add_param_group(
                        {"params": new_rp, "lr": PHASE3_CFG["lr"] * 0.1})
                    dr_existing_ids.update(id(p) for p in new_rp)
                dr_routers_unfrozen = True
                spawn_log.append(step)
                print(f"  [step {step}] Routers unfrozen")

            # ── G: Periodic rank growth (Eq. 9, 12; Algorithm 1 lines 9-14) ───
            if wrapper.schedule.can_grow_at_step(step):
                grow_res = perform_rank_growth(
                    tracker=wrapper.tracker,
                    schedule=wrapper.schedule,
                    lora_modules=wrapper.lora_modules,
                    current_step=step,
                    r_init=PHASE3_CFG["base_rank"] // 4,
                    r_max=PHASE3_CFG["base_rank"],
                    p_grow=0.5,
                    gamma=0.5,
                    verbose=False,
                )
                if grow_res.get("grew"):
                    wrapper.expanded = True
                    spawn_log.append(step)
                    print(f"  [step {step}] Rank growth: +{grow_res['total_new_ranks']} ranks "
                          f"across {grow_res['num_experts_grown']} experts")

        optimizer.step()

        # ── Checkpoint eval ───────────────────────────────────────────────────
        if (step + 1) % PHASE3_CFG["eval_every"] == 0:
            metrics = evaluate_checkpoint(
                wrapper, method_name, run_label, step + 1, PHASE3_CFG)
            metrics["spawn_log"] = list(spawn_log)
            all_results.append(metrics)

    all_run_logs[run_key] = loss_log
    print(f"  Growth/spawn events: {spawn_log}")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("PHASE 3 COMPLETE — FINAL CHECKPOINT RESULTS")
print("=" * 65)
final_rows = [r for r in all_results if r["step"] == PHASE3_CFG["n_steps"]]
run_a_ppl  = {m: None for m in PHASE3_CFG["methods"]}
for r in final_rows:
    if r["run"] == "A":
        run_a_ppl[r["method"]] = r["ppl_code"]

print("{:<18} {:<5} {:>10} {:>13} {:>11} {:>9}".format(
    "Method", "Run", "PPL_code", "NegTransfer", "Expansions", "MeanJSD"))
print("-" * 68)
for r in sorted(final_rows, key=lambda x: (x["method"], x["run"])):
    base = run_a_ppl.get(r["method"])
    neg  = (r["ppl_code"] - base) if base else float("nan")
    print("{:<18} {:<5} {:>10.2f} {:>+13.2f} {:>11}  {:>8.3f}".format(
        r["method"], r["run"], r["ppl_code"], neg,
        r["n_expansions"], r["mean_jsd"]))



METHOD: LORA  |  Run A (conflict=0%)
Loading primary dataset...
    [openai/openai_humaneval] using split='test' (available: ['test'])
Loading conflict dataset...
    [qiaojin/PubMedQA] using split='train' (available: ['train'])

Dataset built: 164 total (164 code / 0 medical)
  [lora | Run A | step 100]  ppl_code=2522.87  ppl_medical=49403.96  mean_jsd=nan  expansions=0
  [lora | Run A | step 200]  ppl_code=1753.44  ppl_medical=60623.96  mean_jsd=nan  expansions=0
  [lora | Run A | step 300]  ppl_code=1477.05  ppl_medical=57256.67  mean_jsd=nan  expansions=0
  [lora | Run A | step 400]  ppl_code=1052.22  ppl_medical=57416.17  mean_jsd=nan  expansions=0
  [lora | Run A | step 500]  ppl_code=1020.75  ppl_medical=54817.41  mean_jsd=nan  expansions=0
  [lora | Run A | step 600]  ppl_code=878.88  ppl_medical=65048.51  mean_jsd=nan  expansions=0
  [lora | Run A | step 700]  ppl_code=635.11  ppl_medical=50927.95  mean_jsd=nan  expansions=0
  [lora | Run A | step 800]  ppl_code=716.67  ppl_m

## Section 10 — Eager Spawn Ablation

In [53]:
# ── Eager Spawn Ablation: Hierarchical Run C with lower delta_threshold ────
# Tests whether earlier spawn timing recovers Run C performance.
# All settings identical to Phase 3 except delta_threshold=0.5 and max_sub_adapters=15.

EAGER_CFG = {**PHASE3_CFG, "delta_threshold": 0.5, "max_sub_adapters": 15}

gc.collect(); torch.cuda.empty_cache()

enc, domains = build_calibration_dataloader(
    tokenizer,
    conflict_ratio=0.5,
    max_length=PHASE3_CFG["max_length"],
    n_each=PHASE3_CFG["n_steps"] + 50,
)

wrapper, optimizer, monitor = setup_method("hierarchical", EAGER_CFG)

eager_results = []
spawn_log = []

for step in range(EAGER_CFG["n_steps"]):
    idx       = step % len(domains)
    input_ids = enc['input_ids'][idx].unsqueeze(0)
    labels    = input_ids.clone()
    labels[:, :-1] = input_ids[:, 1:]
    labels[:, -1]  = -100

    optimizer.zero_grad()
    loss = model_phi(input_ids=input_ids, labels=labels).loss
    loss.backward()

    lora   = wrapper.base_loras[EAGER_CFG["target_expert"]]
    domain = domains[idx]

    if lora.A.grad is not None and lora.B.grad is not None:
        triggered = monitor.update(
            lora_A=lora.A.data, lora_B=lora.B.data,
            loss_val=loss.item(), domain=domain,
        )
        if (triggered and
                len(wrapper.spawn_gates[EAGER_CFG["target_expert"]])
                < EAGER_CFG["max_sub_adapters"]):
            grad_A, grad_B = lora.A.grad, lora.B.grad
            wg = (grad_B.detach().float() @ grad_A.detach().float()
                  if grad_A is not None else None)
            new_params = wrapper.spawn(
                EAGER_CFG["target_expert"], rank=8, weight_grad=wg)
            optimizer.add_param_group(
                {'params': new_params, 'lr': EAGER_CFG["lr"]})
            monitor.reset_after_spawn()
            spawn_log.append(step)

    optimizer.step()

    if (step + 1) % EAGER_CFG["eval_every"] == 0:
        metrics = evaluate_checkpoint(
            wrapper, "hierarchical_eager", "C", step + 1, EAGER_CFG)
        eager_results.append(metrics)

print(f"\nEager spawn events: {spawn_log}")
print(f"First spawn at step: {spawn_log[0] if spawn_log else 'none'}")
print(f"Compare to standard Run C first spawn: 234")
print(f"Final PPL: {eager_results[-1]['ppl_code']:.2f}")
print(f"Compare to standard Run C final PPL:  579.59")

Loading primary dataset...
    [openai/openai_humaneval] using split='test' (available: ['test'])
Loading conflict dataset...
    [qiaojin/PubMedQA] using split='train' (available: ['train'])

Dataset built: 939 total (164 code / 775 medical)
  [hierarchical_eager | Run C | step 100]  ppl_code=2884.95  ppl_medical=62052.62  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 200]  ppl_code=2365.77  ppl_medical=51853.43  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 300]  ppl_code=2075.51  ppl_medical=53519.85  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 400]  ppl_code=1908.57  ppl_medical=53649.95  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 500]  ppl_code=1948.97  ppl_medical=47948.67  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 600]  ppl_code=2130.49  ppl_medical=50060.79  mean_jsd=nan  expansions=0
  [hierarchical_eager | Run C | step 700]  ppl_code=2018.61  ppl_medical=53800.54  mean_jsd=nan  

## Section 11 — Change 2: Multi-Layer Hierarchical Spawning

**Run after Phase 3** (requires `all_results`, `EVAL_CODE_TEXTS`, `EVAL_MEDICAL_TEXTS`).


In [54]:
# ═══════════════════════════════════════════════════════════════════════════
# CHANGE 2: Multi-Layer Hierarchical Spawning (Top-K Jaccard Layers)
# ═══════════════════════════════════════════════════════════════════════════
# PURPOSE
# -------
# The original Phase 3 patches only Layer 0, Expert 0.
# Claim: "gradient-space entanglement is a universal property of shared
# LoRA adapters" — but the evidence is hyper-local to one expert.
#
# This cell extends spawning to the top-K highest-overlap layers from the
# Jaccard diagnostic and runs a new Hierarchical Run B comparison.
# If gains stack with coverage (more layers → less negative transfer),
# that directly validates the mechanistic claim at model scale.
#
# DESIGN CHOICES
# --------------
# - Patch the top-3 layers by per-layer Jaccard score (recomputed below
#   from the router logit data collected in Cell 12).
# - One ConflictSaturationMonitor per patched expert — independent triggers.
# - All monitors share the same hyperparameters locked in Phase 2.
# - Comparison: single-layer Hierarchical Run B vs. multi-layer Run B.
#   Keep everything else identical (same dataset, same optimizer, same steps).
#
# EXPECTED OUTCOME
# ----------------
# If multi-layer outperforms single-layer → entanglement is distributed
# across layers, single-expert framing was conservative, thesis strengthened.
# If multi-layer ≈ single-layer → Layer 0 Expert 0 captures most of the
# conflict, which itself is an interesting finding (localisation of entanglement).
# Either outcome is publishable. Neither is a failure.
# ═══════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import gc
from collections import defaultdict
from accelerate.hooks import remove_hook_from_module, add_hook_to_module

# ── Step 1: Identify top-K layers by Jaccard overlap ─────────────────────
# Re-run routing probe on the Python vs Medical pair and collect
# per-layer shared-expert counts. Reuses get_top_k_experts_for_domain()
# from Cell 12.

TOP_K_LAYERS      = 3     # Number of layers to patch
CONCENTRATION_CUT = 0.002 # Same as Jaccard diagnostic

def compute_per_layer_jaccard(model, tokenizer,
                               primary_texts, conflict_texts,
                               n_examples=60, max_length=128,
                               concentration_cutoff=CONCENTRATION_CUT):
    """
    Returns dict: layer_idx → weighted_jaccard_score.
    Uses the same frequency-weighted metric as Cell 12.
    """
    print("Computing per-layer Jaccard for layer selection...")
    freq_primary  = get_top_k_experts_for_domain(
        model, tokenizer, primary_texts,
        n_examples=n_examples, max_length=max_length)
    freq_conflict = get_top_k_experts_for_domain(
        model, tokenizer, conflict_texts,
        n_examples=n_examples, max_length=max_length)

    # Group by layer
    layers_primary  = defaultdict(dict)  # layer -> {expert: freq}
    layers_conflict = defaultdict(dict)
    for (layer, expert), freq in freq_primary.items():
        layers_primary[layer][expert] = freq
    for (layer, expert), freq in freq_conflict.items():
        layers_conflict[layer][expert] = freq

    all_layers = sorted(set(layers_primary.keys()) | set(layers_conflict.keys()))
    per_layer_jaccard = {}

    for layer in all_layers:
        p = layers_primary.get(layer, {})
        q = layers_conflict.get(layer, {})

        # Only count experts with disproportionate concentration
        conc_p = {e for e, f in p.items() if f > concentration_cutoff}
        conc_q = {e for e, f in q.items() if f > concentration_cutoff}

        shared = conc_p & conc_q
        total  = conc_p | conc_q

        if not total:
            per_layer_jaccard[layer] = 0.0
            continue

        # Frequency-weighted Jaccard
        all_experts = total
        w_intersect = sum(min(p.get(e, 0), q.get(e, 0)) for e in shared)
        w_union     = sum(max(p.get(e, 0), q.get(e, 0)) for e in all_experts)
        per_layer_jaccard[layer] = w_intersect / (w_union + 1e-12)

    return per_layer_jaccard


# Compute (reuses probe texts from Phase 3 setup)
# EVAL_CODE_TEXTS and EVAL_MEDICAL_TEXTS must be defined from Cell 17/18.
layer_jaccards = compute_per_layer_jaccard(
    model_phi, tokenizer,
    EVAL_CODE_TEXTS, EVAL_MEDICAL_TEXTS
)

# Rank and select top-K
sorted_layers = sorted(layer_jaccards.items(), key=lambda x: x[1], reverse=True)
target_layers = [layer for layer, score in sorted_layers[:TOP_K_LAYERS]]

print("\nPer-layer Jaccard scores (top 10):")
for layer, score in sorted_layers[:10]:
    marker = " ◀ SELECTED" if layer in target_layers else ""
    print(f"  Layer {layer:2d}: {score:.4f}{marker}")
print(f"\nSelected layers for multi-layer patch: {target_layers}")


# ── Step 2: Multi-layer patch helper ──────────────────────────────────────

def patch_multiple_layers(model, target_layers, base_rank=16, lora_alpha=32):
    """
    Patch each layer in target_layers with HierarchicalPhimoeExperts.
    Returns dict: layer_idx → (wrapper, ConflictSaturationMonitor)
    """
    wrappers  = {}
    monitors  = {}
    new_params_all = []

    for layer_idx in target_layers:
        current = model.model.layers[layer_idx].mlp.experts

        # Unwrap if already patched
        while hasattr(current, 'original'):
            orig = current.original
            if hasattr(current, '_hf_hook'):
                h = current._hf_hook
                remove_hook_from_module(current)
                add_hook_to_module(orig, h)
            model.model.layers[layer_idx].mlp.experts = orig
            current = orig

        wrapper = HierarchicalPhimoeExperts(
            current, base_rank=base_rank, lora_alpha=lora_alpha
        )
        model.model.layers[layer_idx].mlp.experts = wrapper

        if hasattr(current, '_hf_hook'):
            h = current._hf_hook
            remove_hook_from_module(current)
            add_hook_to_module(wrapper, h)

        monitor = ConflictSaturationMonitor(
            tau_plateau=PHASE3_CFG['tau_plateau'],
            delta_threshold=PHASE3_CFG['delta_threshold'],
            window=15,
        )

        wrappers[layer_idx] = wrapper
        monitors[layer_idx] = monitor
        new_params_all.extend(
            [p for p in wrapper.parameters() if p.requires_grad]
        )
        print(f"  Patched Layer {layer_idx} — "
              f"{sum(p.numel() for p in wrapper.parameters() if p.requires_grad):,} trainable params")

    return wrappers, monitors, new_params_all


def unpatch_all_layers(model, wrappers):
    """Restore all patched layers to original experts."""
    for layer_idx, wrapper in wrappers.items():
        orig = wrapper.original
        if hasattr(wrapper, '_hf_hook'):
            h = wrapper._hf_hook
            remove_hook_from_module(wrapper)
            add_hook_to_module(orig, h)
        model.model.layers[layer_idx].mlp.experts = orig


# ── Step 3: Multi-layer Run B experiment ──────────────────────────────────
# Conflict ratio 0.2 (Run B) — the condition where single-layer already wins.
# Direct comparison: single-layer NegTransfer vs. multi-layer NegTransfer.

print("\n" + "=" * 65)
print("MULTI-LAYER RUN B: Hierarchical Spawning on Top-3 Layers")
print("=" * 65)

gc.collect()
torch.cuda.empty_cache()

# Build Run B dataset (20% conflict)
enc_ml, domains_ml = build_calibration_dataloader(
    tokenizer,
    conflict_ratio=0.2,
    max_length=PHASE3_CFG['max_length'],
    n_each=PHASE3_CFG['n_steps'] + 50,
)

# Patch top-K layers
wrappers_ml, monitors_ml, all_params = patch_multiple_layers(
    model_phi, target_layers,
    base_rank=PHASE3_CFG['base_rank'],
    lora_alpha=PHASE3_CFG['base_rank'] * 2,
)

optimizer_ml = torch.optim.AdamW(all_params, lr=PHASE3_CFG['lr'])
ml_results   = []
spawn_logs_ml = defaultdict(list)  # layer -> [spawn_step, ...]

print(f"Total trainable params across {len(target_layers)} layers: "
      f"{sum(p.numel() for p in all_params):,}")

model_phi.config.use_cache = False
model_phi.train()

for step in range(PHASE3_CFG['n_steps']):
    idx       = step % len(domains_ml)
    input_ids = enc_ml['input_ids'][idx].unsqueeze(0)
    labels    = input_ids.clone()
    labels[:, :-1] = input_ids[:, 1:]
    labels[:, -1]  = -100

    optimizer_ml.zero_grad()
    loss = model_phi(input_ids=input_ids, labels=labels).loss
    loss.backward()

    domain = domains_ml[idx]

    # ── Per-layer spawn check ─────────────────────────────────────────────
    for layer_idx in target_layers:
        wrapper = wrappers_ml[layer_idx]
        monitor = monitors_ml[layer_idx]
        lora    = wrapper.base_loras[0]  # Expert 0 in each layer

        if lora.A.grad is None or lora.B.grad is None:
            continue

        triggered = monitor.update(
            lora_A=lora.A.data,
            lora_B=lora.B.data,
            loss_val=loss.item(),
            domain=domain,
        )

        n_spawned = len(wrapper.spawn_gates[0])
        if triggered and n_spawned < PHASE3_CFG['max_sub_adapters']:
            wg = (lora.B.grad.detach().float() @ lora.A.grad.detach().float()
                  if lora.A.grad is not None else None)
            new_params = wrapper.spawn(0, rank=8, weight_grad=wg)
            optimizer_ml.add_param_group(
                {'params': new_params, 'lr': PHASE3_CFG['lr']}
            )
            monitor.reset_after_spawn()
            spawn_logs_ml[layer_idx].append(step)
            print(f"  [step {step}] Spawn in Layer {layer_idx} "
                  f"(total spawns this layer: {n_spawned + 1})")

    optimizer_ml.step()

    # ── Checkpoint eval ───────────────────────────────────────────────────
    if (step + 1) % PHASE3_CFG['eval_every'] == 0:
        # Use Layer 0 wrapper as representative for JSD (others are bonus)
        main_wrapper = wrappers_ml[target_layers[0]]
        metrics = evaluate_checkpoint(
            main_wrapper, 'hierarchical_multilayer', 'B', step + 1, PHASE3_CFG
        )
        # Log total spawns across all layers
        total_spawns = sum(len(v) for v in spawn_logs_ml.values())
        metrics['total_spawns_all_layers'] = total_spawns
        metrics['spawn_log_per_layer'] = dict(spawn_logs_ml)
        ml_results.append(metrics)

# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("MULTI-LAYER RUN B — FINAL RESULTS")
print("=" * 65)
if ml_results:
    final = ml_results[-1]
    print(f"  Final PPL (code)      : {final['ppl_code']:.2f}")
    print(f"  Spawns per layer      : { {l: len(s) for l, s in spawn_logs_ml.items()} }")

    # Pull single-layer Run B result for comparison
    # (assumes all_results is populated from Phase 3 Cell 21)
    single_layer_runB = [
        r for r in all_results
        if r.get('method') == 'hierarchical' and r.get('run') == 'B'
        and r.get('step') == PHASE3_CFG['n_steps']
    ]
    if single_layer_runB:
        sl_ppl = single_layer_runB[0]['ppl_code']
        sl_nega = single_layer_runB[0].get('ppl_code', float('nan'))
        print(f"  Single-layer PPL (code): {sl_ppl:.2f}")
        diff = final['ppl_code'] - sl_ppl
        direction = 'BETTER' if diff < 0 else 'WORSE'
        print(f"  Multi vs Single delta  : {diff:+.2f}  ({direction})")
        print()
        if diff < 0:
            print("  INTERPRETATION: Multi-layer spawning reduces negative transfer further.")
            print("  Entanglement is distributed across layers — single-expert framing was")
            print("  conservative. Include in thesis as Section 7.x (multi-layer extension).")
        else:
            print("  INTERPRETATION: Multi-layer doesn't help beyond Layer 0.")
            print("  Entanglement may be localised — Layer 0 captures the dominant conflict.")
            print("  This is still a useful finding: the method doesn't degrade with more layers.")
    else:
        print("  (Single-layer Run B results not found in all_results — run Phase 3 first.)")

# ── Teardown ──────────────────────────────────────────────────────────────
unpatch_all_layers(model_phi, wrappers_ml)
gc.collect()
torch.cuda.empty_cache()
print("\nAll layers restored to original state.")


Computing per-layer Jaccard for layer selection...

Per-layer Jaccard scores (top 10):
  Layer 26: 0.5513 ◀ SELECTED
  Layer 14: 0.5025 ◀ SELECTED
  Layer  9: 0.5000 ◀ SELECTED
  Layer  5: 0.4518
  Layer  0: 0.3916
  Layer  6: 0.3737
  Layer 21: 0.3676
  Layer 25: 0.3660
  Layer 31: 0.3565
  Layer 22: 0.3561

Selected layers for multi-layer patch: [26, 14, 9]

MULTI-LAYER RUN B: Hierarchical Spawning on Top-3 Layers
Loading primary dataset...
    [openai/openai_humaneval] using split='test' (available: ['test'])
Loading conflict dataset...
    [qiaojin/PubMedQA] using split='train' (available: ['train'])

Dataset built: 474 total (164 code / 310 medical)
  Patched Layer 26 — 2,097,152 trainable params
  Patched Layer 14 — 2,097,152 trainable params
  Patched Layer 9 — 2,097,152 trainable params
Total trainable params across 3 layers: 6,291,456
  [hierarchical_multilayer | Run B | step 100]  ppl_code=767.71  ppl_medical=48041.04  mean_jsd=nan  expansions=0
  [step 138] Spawn in Layer 14

## Section 12 — DR-LoRA Standalone Full Run (MetaMathQA)

Optional standalone DR-LoRA training on MetaMathQA for comparison with the full saliency pipeline.

In [ ]:
from datasets import load_dataset

# ══════════════════════════════════════════════════════════════════════════════
# MASTER SWITCH
# True  → quick validation : 10 train / 3 test  (code path check, < 1 min)
# False → full experiment  : 4 000 train / 500 test  (~3–4 h on T4)
TEST_MODE = False
# ══════════════════════════════════════════════════════════════════════════════

# Full-mode dataset sizes (only used when TEST_MODE = False)
# Fits within a single 4-hour Kaggle GPU session on a T4:
#   4 000 samples × 3 epochs / batch 4  =  3 000 steps  ≈  3–4 h
N_TRAIN_FULL = 4_000
N_TEST_FULL  =   500   # 12.5 % of training; enough for stable accuracy estimate

print("=" * 80)
print("LOADING METAMATHQA DATASET")
print("=" * 80)

# Load meta-math/MetaMathQA (train split only — no separate test split in HF)
print("\nLoading meta-math/MetaMathQA...")
try:
    full_dataset = load_dataset("meta-math/MetaMathQA", split="train")
    full_dataset = full_dataset.shuffle(seed=42)
    n_total = len(full_dataset)

    if TEST_MODE:
        n_train = 10
        n_test  = 3
    else:
        n_train = min(N_TRAIN_FULL, n_total - N_TEST_FULL)
        n_test  = N_TEST_FULL

    train_samples = list(full_dataset.select(range(n_train)))
    test_samples  = list(full_dataset.select(range(n_train, n_train + n_test)))

    print(f"\nMode: {'TEST' if TEST_MODE else 'FULL'}")
    print(f"  Dataset total:    {n_total:,}")
    print(f"  Training samples: {n_train:,}")
    print(f"  Test samples:     {n_test:,}")

    example = train_samples[0]
    print("\nExample sample fields:", list(example.keys()))
    print(f"  type:     {example['type']}")
    print(f"  query:    {example['query'][:150]}...")
    print(f"  response: {example['response'][:150]}...")

except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Using dummy data for demonstration...")
    n_train = 10 if TEST_MODE else N_TRAIN_FULL
    n_test  = 3  if TEST_MODE else N_TEST_FULL
    train_samples = [
        {"query": f"What is {i}+{i}?", "response": f"The answer is {i*2}.", "type": "dummy"}
        for i in range(n_train)
    ]
    test_samples = [
        {"query": "What is 5+5?", "response": "The answer is 10.", "type": "dummy"}
        for _ in range(n_test)
    ]

print("\n" + "=" * 80)
print("DATASET READY")
print("=" * 80)

# === DATA PREPARATION ===

# Ensure tokenizer has a pad token (required for batched tokenization)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print("Set pad_token = eos_token")


def prepare_batch(samples_batch, tokenizer, max_length=512):
    """
    Prepare a training batch from MetaMathQA samples.

    Key design decisions:
    - Uses tokenizer.apply_chat_template (instruct model format)
    - Labels mask the question with -100 (loss computed on response only)
    - Pads to uniform length within batch

    Args:
        samples_batch: List of MetaMathQA dicts with 'query' and 'response'
        tokenizer: Model tokenizer
        max_length: Max sequence length (truncated if longer)

    Returns:
        Dict with input_ids, attention_mask, labels (all CPU tensors)
    """
    input_ids_list = []
    labels_list = []

    for sample in samples_batch:
        prompt_messages = [{"role": "user", "content": sample["query"]}]
        full_messages   = prompt_messages + [{"role": "assistant", "content": sample["response"]}]

        try:
            prompt_text = tokenizer.apply_chat_template(
                prompt_messages, tokenize=False, add_generation_prompt=True
            )
            full_text = tokenizer.apply_chat_template(
                full_messages, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            # Fallback plain format if chat template unavailable
            prompt_text = f"Question: {sample['query']}\nAnswer: "
            full_text   = prompt_text + sample["response"]

        # Tokenize prompt separately to find the response boundary
        prompt_ids = tokenizer(
            prompt_text, add_special_tokens=False, return_tensors="pt"
        )["input_ids"][0]

        # Tokenize full sequence
        full_enc = tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        full_ids = full_enc["input_ids"][0]

        # Clamp prompt_len so at least 1 response token is supervised (avoids NaN loss)
        prompt_len = min(len(prompt_ids), len(full_ids) - 1)

        # Labels: -100 for question tokens, real ids for response tokens
        labels = full_ids.clone()
        labels[:prompt_len] = -100

        input_ids_list.append(full_ids)
        labels_list.append(labels)

    # Pad to max length in batch
    max_len = max(ids.size(0) for ids in input_ids_list)

    padded_input_ids = []
    padded_labels    = []
    attention_masks  = []

    for ids, lbl in zip(input_ids_list, labels_list):
        pad_len = max_len - ids.size(0)

        padded_input_ids.append(torch.cat([
            ids, torch.full((pad_len,), tokenizer.pad_token_id, dtype=torch.long)
        ]))
        padded_labels.append(torch.cat([
            lbl, torch.full((pad_len,), -100, dtype=torch.long)
        ]))
        attention_masks.append(torch.cat([
            torch.ones(ids.size(0), dtype=torch.long),
            torch.zeros(pad_len, dtype=torch.long)
        ]))

    return {
        "input_ids":      torch.stack(padded_input_ids),
        "attention_mask": torch.stack(attention_masks),
        "labels":         torch.stack(padded_labels),
    }


print("\nData preparation functions ready")
print(f"  Chat template:  {'enabled' if hasattr(tokenizer, 'apply_chat_template') else 'fallback format'}")
print(f"  Label masking:  question tokens masked with -100 (loss on response only)")
print(f"  Pad token:      '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

# Quick verification batch
test_batch = prepare_batch(train_samples[:2], tokenizer, max_length=256)
print(f"\nBatch verification:")
print(f"  input_ids shape:              {test_batch['input_ids'].shape}")
print(f"  labels shape:                 {test_batch['labels'].shape}")
print(f"  masked tokens (question):     {(test_batch['labels'] == -100).sum().item()}")
print(f"  supervised tokens (response): {(test_batch['labels'] != -100).sum().item()}")


In [ ]:
import math
import gc

# ── Configuration ──────────────────────────────────────────────────────────────
FULL_BATCH_SIZE  = 2 if TEST_MODE else 4
MAX_LENGTH       = 256
LEARNING_RATE    = 2e-4
WEIGHT_DECAY     = 0.01
GRAD_CLIP        = 1.0

# DR-LoRA hyperparameters
P_GROW  = 0.5    # max growth fraction per expert per event (Eq. 12)
GAMMA   = 0.5    # rank penalty exponent (Eq. 9)
EMA_BETA = 0.9

# Compute step counts from dataset size
steps_per_epoch = max(1, len(train_samples) // FULL_BATCH_SIZE)
NUM_EPOCHS      = 10 if TEST_MODE else 3
TOTAL_STEPS     = steps_per_epoch * NUM_EPOCHS
WARMUP_STEPS    = max(2,  TOTAL_STEPS // 10)
GROWTH_INTERVAL = max(2,  TOTAL_STEPS // 10)
END_BUFFER      = max(2,  TOTAL_STEPS // 10)
LOG_EVERY       = max(1,  TOTAL_STEPS // 20)

print("=" * 80)
print("STEP 6: FULL DR-LORA TRAINING")
print("=" * 80)
print(f"\nConfiguration:")
print(f"  TEST_MODE:        {TEST_MODE}")
print(f"  train / test:     {len(train_samples)} / {len(test_samples)}")
print(f"  batch_size:       {FULL_BATCH_SIZE}")
print(f"  total_steps:      {TOTAL_STEPS}  ({NUM_EPOCHS} epochs)")
print(f"  warmup_steps:     {WARMUP_STEPS}")
print(f"  growth_interval:  {GROWTH_INTERVAL}")
print(f"  end_buffer:       {END_BUFFER}")
print(f"  r_init / r_target / r_max: {R_INIT} / {R_TARGET} / {R_MAX}")

# ── Reset DR-LoRA for a clean run ──────────────────────────────────────────────
print("\nResetting DR-LoRA weights and masks...")
for info in lora_modules:
    dl = info["dr_lora"]
    nn.init.kaiming_uniform_(dl.lora_A, a=math.sqrt(5))
    nn.init.zeros_(dl.lora_B)
    dl.rank_mask.zero_()
    dl.rank_mask[:R_INIT] = True
    dl.active_ranks = R_INIT
print(f"  Reset {len(lora_modules)} modules to r_init={R_INIT}")

# Re-freeze router (it may have been unfrozen by the demo loop)
freeze_router(model)

# ── Growth schedule ────────────────────────────────────────────────────────────
full_schedule = DRLoRAGrowthSchedule(
    total_steps=TOTAL_STEPS,
    warmup_steps=WARMUP_STEPS,
    growth_interval=GROWTH_INTERVAL,
    r_init=R_INIT,
    r_target=R_TARGET,
    r_max=R_MAX,
    num_experts_per_layer=16,
    num_layers=32,
    end_buffer_steps=END_BUFFER,
)
print(f"\nGrowth schedule: {full_schedule.num_growth_events} events, "
      f"quota={full_schedule.event_quotas[0] if full_schedule.event_quotas else 0}/layer/event")

# ── Tracker with fresh routing hooks ──────────────────────────────────────────
# Remove any stale hooks from previous tracker instances
tracker._remove_routing_hooks()

full_tracker = DRLoRATracker(
    num_layers=32, num_experts_per_layer=16,
    ema_beta=EMA_BETA,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
num_hooks = full_tracker.register_routing_hooks(model)
print(f"Registered {num_hooks} routing hooks")

# ── Optimizer ─────────────────────────────────────────────────────────────────
# Collect only the LoRA params that are currently trainable
lora_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(lora_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(f"Optimizer: {sum(p.numel() for p in lora_params):,} trainable LoRA params")

# ── Training loop ──────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("TRAINING")
print("=" * 80)

model.train()
routers_unfrozen   = False
existing_param_ids = set(id(p) for pg in optimizer.param_groups for p in pg["params"])

train_losses    = []
rank_history    = []
growth_log      = []
data_idx        = 0

for step in range(1, TOTAL_STEPS + 1):

    # ── Batch ─────────────────────────────────────────────────────────────────
    batch_samples = [train_samples[(data_idx + k) % len(train_samples)]
                     for k in range(FULL_BATCH_SIZE)]
    data_idx += FULL_BATCH_SIZE
    batch = prepare_batch(batch_samples, tokenizer, max_length=MAX_LENGTH)
    batch = {k: v.to(model.device) for k, v in batch.items()}

    # ── A: Forward (hooks capture routing weights) ────────────────────────────
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
    )
    loss = outputs.loss

    # ── B: Backward ───────────────────────────────────────────────────────────
    optimizer.zero_grad()
    loss.backward()

    # ── C: Routing frequency EMA (Eq. 5) ─────────────────────────────────────
    full_tracker.update_routing_frequency_from_hooks()

    # ── D: Rank importance EMA (Eq. 6-8) ─────────────────────────────────────
    imp_scores = full_tracker.compute_rank_importance(lora_modules)
    full_tracker.update_rank_importance(imp_scores)

    # ── E: Router unfreeze at warmup (Algorithm 1, line 5-7) ──────────────────
    if step == WARMUP_STEPS and not routers_unfrozen:
        unfreeze_routers(model)
        # Add newly-unfrozen router params to optimizer
        new_router_params = [
            p for p in model.parameters()
            if p.requires_grad and id(p) not in existing_param_ids
        ]
        if new_router_params:
            optimizer.add_param_group({
                "params": new_router_params,
                "lr": LEARNING_RATE * 0.1,   # lower LR for router fine-tuning
                "weight_decay": WEIGHT_DECAY,
            })
            existing_param_ids.update(id(p) for p in new_router_params)
        routers_unfrozen = True

    # ── F: Gradient clip + optimizer step ────────────────────────────────────
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
    optimizer.step()

    # ── G: Periodic rank growth (Eq. 9, 12; Algorithm 1 lines 9-14) ──────────
    if full_schedule.can_grow_at_step(step):
        grow_result = perform_rank_growth(
            tracker=full_tracker, schedule=full_schedule,
            lora_modules=lora_modules, current_step=step,
            r_init=R_INIT, r_max=R_MAX, p_grow=P_GROW, gamma=GAMMA,
        )
        growth_log.append(grow_result)

    # ── Logging ───────────────────────────────────────────────────────────────
    train_losses.append(loss.item())

    if step % LOG_EVERY == 0 or step == 1 or step == TOTAL_STEPS:
        dist = get_rank_distribution(lora_modules)
        rank_history.append({"step": step, **dist})
        total_trainable = sum(p.numel() for pg in optimizer.param_groups
                              for p in pg["params"])
        print(f"  step {step:4d}/{TOTAL_STEPS} | "
              f"loss={loss.item():.4f} | "
              f"r=[{dist['min']}-{dist['max']}] mean={dist['mean']:.2f} | "
              f"router={'ON' if routers_unfrozen else 'off'} | "
              f"trainable={total_trainable:,}")

# ── Training summary ───────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

avg_loss   = sum(train_losses) / len(train_losses)
final_dist = get_rank_distribution(lora_modules)

print(f"\n  Steps completed:    {TOTAL_STEPS}")
print(f"  Avg loss:           {avg_loss:.4f}")
print(f"  Final loss:         {train_losses[-1]:.4f}")
print(f"  Growth events:      {len(growth_log)}")
print(f"  Router unfrozen:    {routers_unfrozen}")
print(f"\n  Final rank distribution:")
print(f"    min:    {final_dist['min']}")
print(f"    max:    {final_dist['max']}")
print(f"    mean:   {final_dist['mean']:.2f}")
print(f"    active LoRA params: {final_dist['active_params']:,}")

if growth_log:
    total_ranks_added = sum(ev["total_new_ranks"] for ev in growth_log)
    print(f"\n  Rank growth summary across {len(growth_log)} events:")
    print(f"    Total new ranks activated: {total_ranks_added}")
    for ev in growth_log:
        print(f"    step {ev['step']:4d}: +"
              f"{ev['total_new_ranks']} ranks  "
              f"({ev['num_experts_grown']} experts, "
              f"{ev['layers_grown']} layers)")

# ── Loss curve ────────────────────────────────────────────────────────────────
if len(train_losses) > 1:
    print(f"\n  Loss trajectory (sampled):")
    n_show = min(10, len(train_losses))
    idxs = [int(i * (len(train_losses) - 1) / (n_show - 1)) for i in range(n_show)]
    for i in idxs:
        bar = "#" * int(train_losses[i] * 20)
        print(f"    step {i+1:4d}: {train_losses[i]:.4f}  {bar}")


In [ ]:
import re


def extract_answer(text: str) -> str:
    """
    Extract the final numerical answer from a MetaMathQA response.
    Tries multiple patterns in order of specificity.
    """
    # 1. LaTeX \boxed{...}
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].strip().replace(",", "")

    # 2. "The answer is X" (case-insensitive)
    m = re.search(r"the answer is[:\s]+([+-]?\d[\d,]*(?:\.\d+)?)", text, re.I)
    if m:
        return m.group(1).replace(",", "")

    # 3. "= X" at end of last sentence
    m = re.search(r"=\s*([+-]?\d[\d,]*(?:\.\d+)?)\s*[.\n]?\s*$", text)
    if m:
        return m.group(1).replace(",", "")

    # 4. Last number in text as fallback
    nums = re.findall(r"[+-]?\d[\d,]*(?:\.\d+)?", text)
    if nums:
        return nums[-1].replace(",", "")

    return text.strip()


def evaluate_dr_lora(model, tokenizer, test_samples, max_new_tokens=512, max_length=512):
    """
    Generate answers on test samples and compute exact-match accuracy.
    """
    model.eval()
    results = []

    with torch.no_grad():
        for sample in test_samples:
            prompt_msgs = [{"role": "user", "content": sample["query"]}]
            try:
                prompt_text = tokenizer.apply_chat_template(
                    prompt_msgs, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                prompt_text = f"Question: {sample['query']}\nAnswer: "

            inputs = tokenizer(
                prompt_text, return_tensors="pt",
                truncation=True, max_length=max_length,
            ).to(model.device)

            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            new_ids   = out_ids[0, inputs["input_ids"].shape[1]:]
            generated = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

            pred_ans  = extract_answer(generated)
            true_ans  = extract_answer(sample["response"])
            correct   = (pred_ans == true_ans)

            results.append({
                "query":     sample["query"],
                "true_resp": sample["response"],
                "generated": generated,
                "true_ans":  true_ans,
                "pred_ans":  pred_ans,
                "correct":   correct,
            })

    model.train()
    return results


# ── Run evaluation ─────────────────────────────────────────────────────────────
print("=" * 80)
print("FINAL EVALUATION ON TEST SET")
print("=" * 80)

print(f"\nEvaluating on {len(test_samples)} held-out test samples...")
eval_results = evaluate_dr_lora(
    model, tokenizer, test_samples,
    max_new_tokens=512, max_length=512,
)

n_correct = sum(r["correct"] for r in eval_results)
accuracy  = n_correct / len(eval_results) if eval_results else 0.0

print(f"\nResults:")
print(f"  Samples evaluated: {len(eval_results)}")
print(f"  Correct:           {n_correct}")
print(f"  Exact-match acc:   {accuracy:.1%}")

# ── Per-sample breakdown ───────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("Sample-level Predictions")
print(f"{'='*80}")

for i, r in enumerate(eval_results):
    print(f"\n[{i+1}/{len(eval_results)}]")
    print(f"  Query:      {r['query'][:120]}")
    print(f"  True ans:   {r['true_ans']}")
    print(f"  Pred ans:   {r['pred_ans']}")
    print(f"  Generated:  {r['generated'][:200]}")
    print(f"  Correct:    {'YES' if r['correct'] else 'NO'}")

# ── Final rank distribution ────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("Final Model State")
print(f"{'='*80}")

final_dist = get_rank_distribution(lora_modules)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  Total model params:          {total_params:,}")
print(f"  Trainable (LoRA+router):     {trainable:,}  ({100*trainable/total_params:.2f}%)")
print(f"  Active LoRA params:          {final_dist['active_params']:,}")
print(f"  Rank distribution:           min={final_dist['min']}  "
      f"max={final_dist['max']}  mean={final_dist['mean']:.2f}")

print(f"\n{'='*80}")
print(f"FINAL ACCURACY:  {accuracy:.1%}  ({n_correct}/{len(eval_results)})")
print(f"{'='*80}")
print("\nDR-LoRA training and evaluation complete.")
print("Steps 1-6 fully implemented:")
print("  [1] Model setup & MoE identification")
print("  [2] DR-LoRA initialization (capacity reservation + binary mask)")
print("  [3] Growth schedule (quota pre-computation)")
print("  [4] Training signal tracking (routing EMA + rank importance)")
print("  [5] Periodic rank growth (saliency-guided greedy allocation)")
print("  [6] Full training loop + final evaluation")
